# Explainable AI-Driven Survival Prediction in Colorectal Cancer — SEER
**Machine-learning *and* survival-analysis pipeline** | Shadab Ansari, M.Tech (CS & IT), MANUU Hyderabad | Guide: Dr. Mohd Omar

Data: **SEER Research Data, 17 Registries, Nov 2025 Sub (2000-2023)**, colon & rectum, diagnosed 2010-2018 (`Survival_Data.csv`).

> **Data-use agreement:** the SEER file is patient-level data. Keep it only on your own computer – do **not** upload it to
> GitHub / Drive / chat. This notebook writes only aggregated results, figures and a trained model.
> Cite SEER (see the *Suggested Citation* in SEER\*Stat) and mention the treatment-data limitations in the paper.


## Roadmap
| Step | What happens |
|---|---|
| 0 | Setup & configuration |
| 1–3 | Load, understand, quality-check the data |
| 4 | Cohort definition (attrition table), outcome definition, **censoring**, leakage control |
| 5 | Feature engineering (harmonise stage/grade/TNM across 2010-2018 coding changes) |
| 6 | EDA, Table 1, feature–outcome association |
| 7 | Train/test split + preprocessing pipeline (feature selection *inside* CV) |
| 8 | Feature-selection comparison: mutual information, L1/Lasso, permutation importance, SHAP |
| 9 | 6 ML models + traditional baselines, stratified k-fold CV, MLflow |
| 10 | Hyper-parameter tuning (RandomizedSearchCV, optional Optuna) |
| 11–12 | Held-out test evaluation, calibration, subgroup performance, bootstrap CIs, sanity check |
| 13 | Sensitivity analysis: with / without pathology and treatment variables |
| 14 | Temporal validation (+ optional external validation) |
| 15 | **Survival models**: Kaplan-Meier, Cox, XGBoost-Cox, (optional RSF / GBSA / CoxNet), IPCW |
| 16 | Explainable AI: SHAP, LIME, partial-dependence plots |
| 17 | Save model + metadata, MLflow |
| 18 | Flask app + Docker + automatic test |
| 19 | Auto-filled abstract numbers + "base-paper limitations → how this notebook fixes them" |

## Step 0 — Setup

In [ ]:
# Run once (remove the #). Optional: optuna, scikit-survival (only needed for the optional cells).
%pip install -q pandas numpy scipy scikit-learn matplotlib seaborn xgboost lightgbm shap lime mlflow lifelines flask gunicorn joblib
%pip install -q optuna scikit-survival

In [ ]:
import os, sys, re, json, time, platform, warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from scipy import stats

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.feature_selection import SelectFromModel, mutual_info_classif
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.cluster import KMeans
from sklearn.isotonic import IsotonicRegression
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from sklearn.calibration import calibration_curve
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, matthews_corrcoef, brier_score_loss,
                             classification_report, ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay)
from xgboost import XGBClassifier, XGBRegressor
from lightgbm import LGBMClassifier

from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import multivariate_logrank_test
from lifelines.utils import concordance_index

import shap
from lime.lime_tabular import LimeTabularExplainer
import mlflow

try:
    import optuna; HAS_OPTUNA = True
except Exception:
    HAS_OPTUNA = False
try:
    from sksurv.util import Surv
    HAS_SKSURV = True
except Exception:
    HAS_SKSURV = False

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110
%matplotlib inline
print("optuna:", HAS_OPTUNA, "| scikit-survival:", HAS_SKSURV)

In [ ]:
# ------------------------------ CONFIGURATION ------------------------------
SEED            = 42
DATA_PATH       = "Survival_Data.csv"       # SEER*Stat export (same folder as this notebook)
HORIZON         = 60                         # 5-year survival = 60 months
FEATURE_SET     = "B"                        # "A" diagnosis+staging | "B" + pathology | "C" + treatment  (see Step 5)
TIMEPOINT       = {"A": "diagnosis / clinical staging", "B": "post-surgical pathological assessment", "C": "after treatment decisions"}
THRESHOLD       = 0.5                        # PRE-SPECIFIED decision threshold on the (uncalibrated) predicted probability of surviving 5 years
TIE_MARGIN      = 0.002                      # model-selection rule, added after the first run (post hoc): highest mean CV ROC-AUC; models within TIE_MARGIN of the best are "tied" ...
TIE_BREAK       = "Brier"                    # ... and a tie is broken by the lowest mean CV Brier score (better calibrated probabilities)
N_BOOT          = 1000                       # bootstrap resamples for 95% confidence intervals
TEMPORAL_CUT    = 2015                       # temporal validation: develop on <= 2015, test on > 2015
TEST_SIZE       = 0.20
CV_FOLDS        = 5
CV_SAMPLE       = 80000                      # rows used for the model-comparison CV (speed)
SVM_SAMPLE      = 15000                      # RBF-SVM is O(n^2): trained on a stratified subsample
USE_FEATURE_SELECTION = True                 # L1/Lasso selection INSIDE the pipeline (no leakage)
DO_TUNING       = True
TUNING_ITER     = 10
TUNING_SAMPLE   = 60000
USE_OPTUNA      = False                      # True -> extra Optuna study for XGBoost (needs optuna)
OPTUNA_TRIALS   = 20
RUN_SKSURV      = True                       # optional RSF / GBSA / CoxNet (needs scikit-survival)
SURV_SAMPLE     = 60000                      # rows for Cox / survival-model fitting (speed)
FINAL_MODEL_OVERRIDE = None                  # e.g. "XGBoost" to force an explainable final model (say so in the paper!)
EXTERNAL_PATH   = "gse39582_clinical.csv"    # optional external-validation file (see Step 14)
FAST_MODE       = False                      # True -> 40k-row sample, quick debugging run
FAST_N          = 40000

if FAST_MODE:
    CV_FOLDS, CV_SAMPLE, SVM_SAMPLE = 3, 12000, 5000
    TUNING_ITER, TUNING_SAMPLE, SURV_SAMPLE, OPTUNA_TRIALS = 3, 12000, 15000, 5
    N_BOOT = 200

OUT = Path("outputs"); FIG = OUT / "figures"; ART = Path("artifacts")
for p in (OUT, FIG, ART): p.mkdir(parents=True, exist_ok=True)
print(f"Primary model: feature set {FEATURE_SET} = prediction at {TIMEPOINT[FEATURE_SET]} | threshold {THRESHOLD} | selection: max CV AUC, ties (<{TIE_MARGIN}) -> min CV {TIE_BREAK}")
def savefig(name): plt.savefig(FIG / f"{name}.png", dpi=200, bbox_inches="tight")

## Step 1 — Load data

In [ ]:
df_raw = pd.read_csv(DATA_PATH, low_memory=False)
if FAST_MODE:
    df_raw = df_raw.sample(min(FAST_N, len(df_raw)), random_state=SEED).reset_index(drop=True)
print("Shape:", df_raw.shape)
df_raw.head(3)

## Step 2 — Data understanding

In [ ]:
display(pd.DataFrame({"dtype": df_raw.dtypes.astype(str), "n_unique": df_raw.nunique(),
                      "n_blank_or_unknown": df_raw.apply(lambda s: int(s.astype(str).str.startswith(("Blank", "Unknown", "UNK", "Not documented")).sum()))}))
print("Memory (MB):", round(df_raw.memory_usage(deep=True).sum() / 1e6, 1))

## Step 3 — Data-quality checks
SEER codes missing information as text (`Blank(s)`, `Unknown`, …) rather than empty cells, so we count those explicitly.
`Patient ID` is **not unique** because one person can have several primary cancers – we handle this in Step 4.

In [ ]:
print("True missing cells:", int(df_raw.isna().sum().sum()), "(only the older AJCC stage columns - expected)")
print("Rows:", len(df_raw), "| unique Patient IDs:", df_raw["Patient ID"].nunique())
print("Exact duplicate rows (ignoring ID):", int(df_raw.drop(columns="Patient ID").duplicated().sum()), "(distinct patients can look identical - NOT dropped)")

all_blank = [c for c in df_raw.columns if df_raw[c].astype(str).str.startswith("Blank").all()]
print("Columns that are 100% blank (dropped):", all_blank)
print("\nSurvival-months flag:"); print(df_raw["Survival months flag"].value_counts().to_string())
print("\nBehaviour:", df_raw["Behavior code ICD-O-3"].unique().tolist(), "| Years:", sorted(df_raw["Year of diagnosis"].unique().tolist()))

## Step 4 — Cohort, outcome, censoring and leakage control

**Cohort (inclusion/exclusion)** – first or only primary cancer, diagnosed 2010-2018, adults (≥ 20 y), not death-certificate/autopsy-only,
known survival time, not *in-situ*. The attrition table below is what goes into the paper's flow-chart.

**Outcome**
* `Time` = SEER *survival months*, `Event` = 1 if dead (all-cause) at the study cut-off.
* **5-year label `y5`:** survived ≥ 60 months → **1**; died before 60 months → **0**;
  *alive with < 60 months follow-up* = **censored → `NaN`** (outcome unknown). These patients are **not** thrown away silently:
  they stay in the survival models (Step 15) and are reported here.
* Diagnosis years end in 2018 so that follow-up to the study cut-off can reach 5 years.

**Leakage control:** `Patient ID`, `Survival months`, `Vital status` are never features;
`Reason no cancer-directed surgery` is excluded because it contains *death-related* categories
("patient died prior to recommended surgery").

In [ ]:
AGE_COL   = "Age recode with <1 year olds and 90+"
STAGE_COL = "Combined Summary Stage with Expanded Regional Codes (2004+)"

def age_to_years(s):
    nums = re.findall(r"\d+", str(s))
    if str(s).startswith("90+") or "+" in str(s): return float(nums[0]) + 2
    if len(nums) >= 2: return (float(nums[0]) + float(nums[1])) / 2
    return float(nums[0]) if nums else np.nan

d = df_raw.drop(columns=all_blank).copy()
attr = [("Exported records", len(d))]

d = d[d["Sequence number"].isin(["One primary only", "1st of 2 or more primaries"])];            attr.append(("First / only primary cancer", len(d)))
d = d[~d["Type of Reporting Source"].isin(["Death certificate only", "Autopsy only"])];            attr.append(("Excl. death-certificate / autopsy-only", len(d)))
d["Time"] = pd.to_numeric(d["Survival months"], errors="coerce")
d = d[d["Time"].notna()];                                                                          attr.append(("Known survival time", len(d)))
d["Age_years"] = d[AGE_COL].map(age_to_years)
d = d[d["Age_years"] >= 20];                                                                       attr.append(("Adults (>= 20 years)", len(d)))
d = d[d[STAGE_COL] != "In situ"];                                                                  attr.append(("Excl. in-situ stage", len(d)))

d["Event"] = d["Vital status recode (study cutoff used)"].eq("Dead").astype(int)
d["y5"] = np.where(d["Time"] >= HORIZON, 1.0, np.where(d["Event"] == 1, 0.0, np.nan))

attrition = pd.DataFrame(attr, columns=["Step", "N"]); attrition["Removed"] = (-attrition["N"].diff()).fillna(0).astype(int)
display(attrition)
attrition.to_csv(OUT / "cohort_attrition.csv", index=False)

n_cens = int(d["y5"].isna().sum())
print(f"\nFinal cohort: {len(d):,} patients | median follow-up {d['Time'].median():.0f} months | deaths {int(d['Event'].sum()):,}")
print(f"5-year label: survived={int((d.y5==1).sum()):,}  died<5y={int((d.y5==0).sum()):,}  "
      f"censored(alive, <60 mo)={n_cens:,} ({100*n_cens/len(d):.1f}%) -> excluded from binary models, kept in survival models")
print(f"Class balance (labelled): {100*d.y5.mean():.1f}% survived 5 years  -> mild imbalance, handled with class weights")

## Step 5 — Feature engineering
SEER changed its coding between 2010 and 2018 (AJCC 7th → SEER-combined → EOD 2018). We **harmonise** the three versions
of stage, T/N/M and grade into one consistent set of columns, convert interval variables to numbers, and keep SEER's
"unknown" as an explicit category (so the model can learn from missingness).

**Three feature sets** (this is the "available at prediction time + sensitivity analysis" requirement):

| Set | Contents | When is it known? |
|---|---|---|
| **A** | demographics, site, histology, grade, stage, T/N/M, tumour size, CEA, income, rurality | diagnosis & staging |
| **B** | A + lymph-node counts, perineural invasion, tumour deposits | after surgical pathology |
| **C** | B + surgery, radiation, chemotherapy | after treatment decisions (SEER treatment data has documented limitations) |

`FEATURE_SET` (Step 0) selects the set used for the main model; Step 13 compares all three.

In [ ]:
BLANKS = ("Blank(s)",)

def first_valid(frame, cols):
    out = pd.Series(np.nan, index=frame.index, dtype=object)
    for c in cols:
        s = frame[c].astype(object)
        ok = out.isna() & s.notna() & ~s.isin(BLANKS)
        out[ok] = s[ok]
    return out

def norm_stage(v):
    s = str(v).strip().upper()
    if s in ("", "NAN", "BLANK(S)", "UNK STAGE", "99", "88"): return "Unknown"
    if s == "0": return "0"
    s = s.replace("NOS", "")
    if s[:1] in "1234" and s[:1] != "": return {"1": "I", "2": "II", "3": "III", "4": "IV"}[s[0]]
    for r in ("IV", "III", "II", "I"):
        if s.startswith(r): return r
    return "Unknown"

def norm_tnm(v, letter):
    s = str(v).strip().upper()
    if s in ("", "NAN", "BLANK(S)", "NOT APPLICABLE", "88"): return "Unknown"
    s = re.sub(rf"^[CP{letter}]", "", s)             # c1A / p2B / T3 / N1a -> 1A / 2B / 3 / 1A
    if letter == "T":
        if s.startswith("IS"): return "Tis"
        return {"0": "T0", "1": "T1", "2": "T2", "3": "T3", "4": "T4"}.get(s[:1], "Unknown")
    if letter == "N": return {"0": "N0", "1": "N1", "2": "N2"}.get(s[:1], "Unknown")
    return {"0": "M0", "1": "M1"}.get(s[:1], "Unknown")

def income_to_num(v):
    s = str(v).replace(",", "")
    if s.startswith("Unknown"): return np.nan
    m = re.findall(r"\d+", s)
    if s.startswith("<") and m: return float(m[0]) - 2500
    if s.endswith("+") and m:   return float(m[0]) + 5000
    if len(m) >= 2:             return (float(m[0]) + float(m[1])) / 2
    return np.nan

def rucc_ord(v):
    s = str(v)
    if s.startswith("Counties in metropolitan areas ge 1 million"): return 0
    if "250,000 to 1 million" in s: return 1
    if "lt 250 thousand" in s: return 2
    if s.startswith("Nonmetropolitan counties adjacent"): return 3
    if s.startswith("Nonmetropolitan counties not adjacent"): return 4
    return np.nan

def tumour_mm(v):
    s = str(v)
    if s.isdigit():
        x = int(s); return float(x) if 1 <= x <= 400 else np.nan
    if s.startswith("990"): return 0.5
    if s.startswith("000"): return 0.0
    return np.nan

GRADE_OLD = {"Well differentiated; Grade I": "I", "Moderately differentiated; Grade II": "II",
             "Poorly differentiated; Grade III": "III", "Undifferentiated; anaplastic; Grade IV": "IV"}
GRADE_NEW = {"Site-specific grade system category (1)": "I", "Site-specific grade system category (2)": "II",
             "Site-specific grade system category (3)": "III", "Site-specific grade system category (4)": "IV",
             "Well differentiated": "I", "Moderately differentiated": "II", "Poorly differentiated": "III",
             "Undifferentiated and anaplastic": "IV"}          # NOTE: 2018+ 'category (n)' assumed = grade n (check SEER dictionary)
STAGE_MAP = {"Localized only": "Localized", "Regional by direct extension only": "Regional (extension)",
             "Regional lymph nodes involved only": "Regional (nodes)",
             "Regional by both direct extension and lymph node involvement": "Regional (both)",
             "Distant site(s)/node(s) involved": "Distant", "Unknown/unstaged/unspecified/DCO": "Unknown"}

C = pd.DataFrame(index=d.index)
C["Patient_ID"], C["Year"], C["Time"], C["Event"], C["y5"] = d["Patient ID"], d["Year of diagnosis"], d["Time"], d["Event"], d["y5"]
C["Time"] = C["Time"].clip(lower=0.5)

# ---- A: diagnosis & staging ----
C["Age_years"]      = d["Age_years"]
C["Sex"]            = d["Sex"]
C["Race"]           = d["Race and origin recode (NHW, NHB, NHAIAN, NHAPI, Hispanic)"].str.replace("Non-Hispanic", "NH")
C["Marital_Status"] = d["Marital status at diagnosis"]
C["Income_USD"]     = d["Median household income inflation adj to 2024"].map(income_to_num)
C["RUCC"]           = d["Rural-Urban Continuum Code"].map(rucc_ord)
C["Site"]           = d["Site recode ICD-O-3/WHO 2008"]
h = d["Histology recode - broad groupings"].str.replace(r"^\d{4}-\d{4}:\s*", "", regex=True)
keep = h.value_counts(normalize=True); keep = keep[keep >= 0.01].index           # label-free grouping of rare histologies
C["Histology"]      = h.where(h.isin(keep), "Other")
C["Grade"]          = d["Grade Recode (thru 2017)"].map(GRADE_OLD).fillna(d["Derived Summary Grade 2018 (2018+)"].map(GRADE_NEW)).fillna("Unknown")
C["Summary_Stage"]  = d[STAGE_COL].map(STAGE_MAP).fillna("Unknown")
C["Stage_Simple"]   = C["Summary_Stage"].map(lambda s: "Regional" if s.startswith("Regional") else s)
C["Stage_Group"]    = first_valid(d, ["Derived AJCC Stage Group, 7th ed (2010-2015)", "7th Edition Stage Group Recode (2016-2017)",
                                      "Derived EOD 2018 Stage Group Recode (2018+)"]).map(norm_stage)
C["T_Stage"] = first_valid(d, ["Derived AJCC T, 7th ed (2010-2015)", "Derived SEER Combined T (2016-2017)", "Derived EOD 2018 T Recode (2018+)"]).map(lambda v: norm_tnm(v, "T"))
C["N_Stage"] = first_valid(d, ["Derived AJCC N, 7th ed (2010-2015)", "Derived SEER Combined N (2016-2017)", "Derived EOD 2018 N Recode (2018+)"]).map(lambda v: norm_tnm(v, "N"))
C["M_Stage"] = first_valid(d, ["Derived AJCC M, 7th ed (2010-2015)", "Derived SEER Combined M (2016-2017)", "Derived EOD 2018 M Recode (2018+)"]).map(lambda v: norm_tnm(v, "M"))
C["Tumor_Size_mm"]  = d["Tumor Size Over Time Recode (1988+)"].map(tumour_mm)
cea = d["CEA Pretreatment Interpretation Recode (2010+)"].astype(str).str.lower()
C["CEA"] = np.select([cea.str.contains("positive"), cea.str.contains("negative"), cea.str.contains("borderline")], ["Positive", "Negative", "Borderline"], "Unknown")

# ---- B: surgical pathology ----
ne = pd.to_numeric(d["Regional nodes examined (1988+)"], errors="coerce");  ne = ne.where(ne.between(0, 90))     # 95-99 = special codes
npos = pd.to_numeric(d["Regional nodes positive (1988+)"], errors="coerce"); npos = npos.where(npos.between(0, 90))  # 95/97/98/99 = special codes
C["Nodes_Examined"], C["Nodes_Positive"] = ne, npos
C["Node_Ratio"] = (npos / ne).where(ne > 0)
pni = d["Perineural Invasion Recode (2010+)"].astype(str).str.lower()
C["Perineural_Invasion"] = np.select([pni.str.contains("identified/present"), pni.str.contains("not identified")], ["Present", "Absent"], "Unknown")
td = d["Tumor Deposits Recode (2010+)"].astype(str)
C["Tumor_Deposits"] = np.select([td.eq("No tumor deposits"), td.str.isdigit() | td.str.startswith("Tumor Deposits identified") | td.str.startswith("81 or more")],
                                ["None", "Present"], "Unknown")

# ---- C: treatment ----
sc = d["RX Summ--Surg Prim Site (1998-2022)"]
C["Surgery"]      = np.select([sc.eq(0), sc.eq(99)], ["No", "Unknown"], "Yes")
C["Radiation"]    = np.where(d["Radiation recode"].astype(str).str.contains("beam|isotope|implant|NOS", case=False, regex=True), "Yes", "No/Unknown")
C["Chemotherapy"] = d["Chemotherapy recode (yes, no/unk)"]

NUM_A = ["Age_years", "Income_USD", "RUCC", "Tumor_Size_mm"]
CAT_A = ["Sex", "Race", "Marital_Status", "Site", "Histology", "Grade", "Summary_Stage", "Stage_Group", "T_Stage", "N_Stage", "M_Stage", "CEA"]
NUM_B = ["Nodes_Examined", "Nodes_Positive", "Node_Ratio"];  CAT_B = ["Perineural_Invasion", "Tumor_Deposits"]
CAT_C = ["Surgery", "Radiation", "Chemotherapy"]

def feats_for(fs):
    num = NUM_A + (NUM_B if fs in ("B", "C") else [])
    cat = CAT_A + (CAT_B if fs in ("B", "C") else []) + (CAT_C if fs == "C" else [])
    return num, cat

NUM_ALL, CAT_ALL = feats_for("C")
for c in CAT_ALL + ["Stage_Simple"]: C[c] = C[c].astype(str).replace({"nan": "Unknown"})
NUM_COLS, CAT_COLS = feats_for(FEATURE_SET)
FEATS = NUM_COLS + CAT_COLS
print(f"Feature set {FEATURE_SET}: {len(NUM_COLS)} numeric + {len(CAT_COLS)} categorical = {len(FEATS)} features")
print("Engineered cohort:", C.shape)
display(C[NUM_ALL].describe().T.round(2))
display(pd.DataFrame({c: C[c].value_counts().head(4).to_dict() for c in ["Stage_Group", "T_Stage", "N_Stage", "M_Stage", "Grade"]}).T)

## Step 6 — EDA, Table 1 and feature–outcome association
Table 1 (patients who survived 5 years vs. died within 5 years) is the standard first table of a clinical paper.
Effect sizes (Cramér's V / rank-biserial) show which variables matter *before* any model is trained.

In [ ]:
lab = C[C.y5.notna()].copy()
rows = []
for c in NUM_ALL:
    a, b = lab.loc[lab.y5 == 1, c].dropna(), lab.loc[lab.y5 == 0, c].dropna()
    p = stats.mannwhitneyu(a, b).pvalue
    f = lambda s: f"{s.median():.1f} ({s.quantile(.25):.1f}-{s.quantile(.75):.1f})"
    rows.append((c, "median (IQR)", f(a), f(b), p))
for c in CAT_ALL:
    ct = pd.crosstab(lab[c], lab.y5.astype(int)); p = stats.chi2_contingency(ct)[1]
    for lvl in ct.index:
        rows.append((f"{c}: {lvl}", "n (%)", f"{ct.loc[lvl, 1]:,} ({100*ct.loc[lvl, 1]/ct[1].sum():.1f}%)", f"{ct.loc[lvl, 0]:,} ({100*ct.loc[lvl, 0]/ct[0].sum():.1f}%)", p))
table1 = pd.DataFrame(rows, columns=["Variable", "Statistic", "Survived >=5y", "Died <5y", "p-value"])
table1.to_csv(OUT / "table1_cohort_characteristics.csv", index=False)
print(f"Table 1: {len(lab):,} patients with a known 5-year outcome (survived {int(lab.y5.sum()):,} / died {int((1-lab.y5).sum()):,})")
display(table1.head(40))

In [ ]:
def cramers_v(x, y):
    ct = pd.crosstab(x, y); chi2, p, _, _ = stats.chi2_contingency(ct, correction=False)
    return np.sqrt(chi2 / (ct.values.sum() * (min(ct.shape) - 1))), p

eff = []
for c in CAT_ALL: v, p = cramers_v(lab[c], lab.y5); eff.append((c, "categorical", v, p))
for c in NUM_ALL:
    a, b = lab.loc[lab.y5 == 1, c].dropna(), lab.loc[lab.y5 == 0, c].dropna()
    u, p = stats.mannwhitneyu(a, b); eff.append((c, "numeric", abs(1 - 2 * u / (len(a) * len(b))), p))
assoc = pd.DataFrame(eff, columns=["feature", "type", "effect_size", "p_value"]).sort_values("effect_size", ascending=False)
assoc.to_csv(OUT / "feature_outcome_association.csv", index=False)
plt.figure(figsize=(7, 6)); sns.barplot(data=assoc.head(18), y="feature", x="effect_size", color="#1d3557")
plt.title("Association with 5-year survival (effect size)"); plt.tight_layout(); savefig("eda_effect_sizes"); plt.show()
display(assoc.head(10).round(4))

In [ ]:
eda_cats = ["Stage_Simple", "Stage_Group", "Grade", "Site", "Sex", "Race", "Marital_Status", "CEA", "N_Stage", "T_Stage", "Perineural_Invasion", "Surgery"]
overall = lab.y5.mean()
fig, axes = plt.subplots(3, 4, figsize=(18, 11))
for ax, c in zip(axes.ravel(), eda_cats):
    r = lab.groupby(c)["y5"].agg(["mean", "size"]); r = r[r["size"] >= 30]
    ax.bar(range(len(r)), r["mean"], color="#457b9d"); ax.axhline(overall, color="red", ls="--", lw=1)
    ax.set_xticks(range(len(r))); ax.set_xticklabels(r.index, rotation=75, ha="right", fontsize=7); ax.set_ylim(0, 1); ax.set_title(c, fontsize=10); ax.grid(False, axis="x")
plt.suptitle("5-year survival rate by category (red = overall)", y=1.0); plt.tight_layout(); savefig("eda_survival_by_category"); plt.show()

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for a_, c in zip(ax, ["Age_years", "Tumor_Size_mm", "Nodes_Positive"]):
    sns.histplot(lab, x=c, hue="y5", element="step", stat="density", common_norm=False, ax=a_, bins=30); a_.set_title(c)
plt.tight_layout(); savefig("eda_numeric_by_outcome"); plt.show()

## Step 7 — Train/test split and the modelling pipeline
* One stratified 80/20 split on **patients** (each patient appears once after the first-primary filter; asserted below).
* `train_all` / `test_all` contain **everyone** (incl. censored) → survival models. `train_b` / `test_b` contain the labelled patients → classifiers.
* Everything that learns from data (imputer, scaler, encoder, **feature selection**) sits inside the `Pipeline`,
  so it is re-fitted on the training folds only → **no feature-selection leakage**.

In [ ]:
assert C["Patient_ID"].is_unique, "Patient IDs must be unique after the first-primary filter"
strat = C["y5"].fillna(-1)
idx_tr, idx_te = train_test_split(C.index, test_size=TEST_SIZE, stratify=strat, random_state=SEED)
train_all, test_all = C.loc[idx_tr], C.loc[idx_te]
train_b, test_b = train_all[train_all.y5.notna()], test_all[test_all.y5.notna()]
ALLX = list(dict.fromkeys(NUM_ALL + CAT_ALL + ["Stage_Simple"]))            # superset of columns kept in X
X_train, y_train = train_b[ALLX], train_b["y5"].astype(int)
X_test,  y_test  = test_b[ALLX],  test_b["y5"].astype(int)
print(f"train_all {len(train_all):,} | test_all {len(test_all):,} | labelled train {len(X_train):,} | labelled test {len(X_test):,}")
print(f"survived-5y rate: train {y_train.mean():.3f} | test {y_test.mean():.3f}")

def make_preprocessor(num_cols, cat_cols):
    num = Pipeline([("imp", SimpleImputer(strategy="median", add_indicator=True)), ("sc", StandardScaler())])
    cat = OneHotEncoder(handle_unknown="ignore", min_frequency=0.005, sparse_output=False)
    return ColumnTransformer([("num", num, num_cols), ("cat", cat, cat_cols)], verbose_feature_names_out=False)

def make_selector():
    if not USE_FEATURE_SELECTION: return "passthrough"
    return SelectFromModel(LogisticRegression(penalty="l1", solver="liblinear", C=0.05, max_iter=300), threshold=1e-6)

def build_pipeline(clf, fs=None):
    num, cat = feats_for(fs or FEATURE_SET)
    return Pipeline([("prep", make_preprocessor(num, cat)), ("select", make_selector()), ("clf", clf)])

def selected_names(pipe):
    names = pipe.named_steps["prep"].get_feature_names_out()
    sel = pipe.named_steps["select"]
    return list(names) if isinstance(sel, str) else list(names[sel.get_support()])

def origin_of(n, cats=CAT_ALL):
    n = n.replace("missingindicator_", "")
    if n in ALLX: return n
    for c in sorted(cats, key=len, reverse=True):
        if n.startswith(c + "_"): return c
    return n

_s = X_train.sample(min(20000, len(X_train)), random_state=SEED)
probe = build_pipeline(LogisticRegression(max_iter=300)).fit(_s, y_train.loc[_s.index])
print(f"After encoding: {len(probe.named_steps['prep'].get_feature_names_out())} columns -> L1 selection keeps {len(selected_names(probe))}")

## Step 8 — Feature-importance methods compared
The base paper used a univariate chi-square filter. Here four *different* views are compared (computed on the training set only):
**mutual information** (non-linear), **L1/Lasso** (linear, multivariate), **permutation importance** (model-based, held-out), **SHAP** (model-based, additive).
One-hot columns are summed back to the original clinical variable.

In [ ]:
n_fs = min(40000, len(X_train))
Xs_raw = X_train.sample(n_fs, random_state=SEED); ys = y_train.loc[Xs_raw.index]
prep_fs = make_preprocessor(NUM_COLS, CAT_COLS).fit(Xs_raw)
Xt = prep_fs.transform(Xs_raw); names = list(prep_fs.get_feature_names_out())
Xa, Xv, ya, yv = train_test_split(Xt, ys.values, test_size=0.25, stratify=ys.values, random_state=SEED)

n_mi = min(15000, len(Xa)); mi = mutual_info_classif(Xa[:n_mi], ya[:n_mi], random_state=SEED)
l1 = np.abs(LogisticRegression(penalty="l1", solver="liblinear", C=0.05, max_iter=300).fit(Xa, ya).coef_[0])
xgb_fs = XGBClassifier(n_estimators=150, max_depth=4, learning_rate=0.1, subsample=0.8, tree_method="hist", n_jobs=-1, random_state=SEED).fit(Xa, ya)
pi = permutation_importance(xgb_fs, Xv[:10000], yv[:10000], scoring="roc_auc", n_repeats=3, random_state=SEED).importances_mean.clip(min=0)
sv_fs = shap.TreeExplainer(xgb_fs).shap_values(Xv[:2000]); sv_fs = sv_fs[1] if isinstance(sv_fs, list) else sv_fs
shap_imp = np.abs(sv_fs).mean(0)

raw = pd.DataFrame({"Mutual information": mi, "L1 / Lasso |coef|": l1, "Permutation importance": pi, "SHAP mean|value|": shap_imp}, index=names)
by_var = raw.groupby([origin_of(n) for n in raw.index]).sum()
share = by_var / by_var.sum()
share["Mean share"] = share.mean(axis=1)
share = share.sort_values("Mean share", ascending=False); share.to_csv(OUT / "feature_importance_methods.csv")
display((100 * share.head(15)).round(1).astype(str) + " %")

top = share.head(12)[::-1].drop(columns="Mean share")
top.plot(kind="barh", figsize=(9, 7), width=0.8); plt.xlabel("share of total importance"); plt.title("Feature importance: four methods compared")
plt.tight_layout(); savefig("feature_importance_methods"); plt.show()

rho = share.drop(columns="Mean share").corr(method="spearman")
plt.figure(figsize=(5, 4)); sns.heatmap(rho, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1); plt.title("Rank agreement between methods (Spearman)")
plt.tight_layout(); savefig("feature_importance_agreement"); plt.show()

## Step 9 — Models, traditional baselines and stratified k-fold cross-validation
* **ML models:** Logistic Regression, Random Forest, Gradient Boosting, XGBoost, LightGBM, SVM (RBF).
* **Baselines a model must beat:** majority class; *traditional clinical* logistic regressions (stage only; stage + age + sex).
* Class imbalance → class weights; metrics → accuracy, balanced accuracy, precision, recall, F1, ROC-AUC, **PR-AUC**, MCC, Brier.
* The comparison CV runs on a stratified sample of `CV_SAMPLE` training rows (speed); final models are trained on the full training set.
* Every run is logged to **MLflow**.
* **Model-selection rule (set in Step 0 after the first run; post hoc):** the model with the highest mean CV ROC-AUC is selected; models whose mean CV ROC-AUC is within `TIE_MARGIN` of the best are considered tied and the tie is broken by the lowest mean CV Brier score. The held-out test set is never used for selection.

In [ ]:
def get_scores(model, X_):
    return model.predict_proba(X_)[:, 1] if hasattr(model, "predict_proba") else model.decision_function(X_)

def predict_label(model, X_):
    """Class label with the PRE-SPECIFIED threshold (identical to model.predict() when THRESHOLD = 0.5)."""
    if hasattr(model, "predict_proba"):
        return (model.predict_proba(X_)[:, 1] >= THRESHOLD).astype(int)
    return model.predict(X_)                                   # SVM: sign of the decision function

def evaluate(model, X_, y_):
    pred, s = predict_label(model, X_), get_scores(model, X_)
    return {"Accuracy": accuracy_score(y_, pred), "Balanced Acc.": balanced_accuracy_score(y_, pred),
            "Precision": precision_score(y_, pred, zero_division=0), "Recall": recall_score(y_, pred),
            "F1-score": f1_score(y_, pred), "ROC-AUC": roc_auc_score(y_, s), "PR-AUC": average_precision_score(y_, s),
            "MCC": matthews_corrcoef(y_, pred), "Brier": brier_score_loss(y_, s) if hasattr(model, "predict_proba") else np.nan}

neg, pos = int((y_train == 0).sum()), int((y_train == 1).sum())
models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED),
    "Random Forest":       RandomForestClassifier(n_estimators=300, max_depth=14, min_samples_leaf=30, class_weight="balanced_subsample", n_jobs=-1, random_state=SEED),
    "Gradient Boosting":   GradientBoostingClassifier(n_estimators=150, learning_rate=0.1, max_depth=3, subsample=0.8, random_state=SEED),
    "XGBoost":             XGBClassifier(n_estimators=400, learning_rate=0.05, max_depth=5, subsample=0.8, colsample_bytree=0.8,
                                         scale_pos_weight=neg / pos, eval_metric="logloss", tree_method="hist", n_jobs=-1, random_state=SEED),
    "LightGBM":            LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=31, subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
                                          class_weight="balanced", verbose=-1, n_jobs=-1, random_state=SEED),
    "SVM (RBF)":           SVC(kernel="rbf", C=1.0, gamma="scale", class_weight="balanced", random_state=SEED),
}
BASELINE = "Baseline (majority class)"
TRAD = {"Traditional: Stage only (LR)": (["Stage_Simple"], []),
        "Traditional: Stage + Age + Sex (LR)": (["Stage_Simple", "Sex"], ["Age_years"])}

def trad_pipeline(cat, num):
    ct = ColumnTransformer([("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num),
                            ("cat", OneHotEncoder(handle_unknown="ignore"), cat)])
    return Pipeline([("prep", ct), ("clf", LogisticRegression(max_iter=1000))])

def make_model(name):
    if name == BASELINE: return build_pipeline(DummyClassifier(strategy="prior"))
    if name in TRAD:     return trad_pipeline(*TRAD[name])
    return build_pipeline(clone(models[name]))

def sub_stratified(X_, y_, n):
    if len(X_) <= n: return X_, y_
    Xs_, _, ys_, _ = train_test_split(X_, y_, train_size=n, stratify=y_, random_state=SEED)
    return Xs_, ys_

def svm_sub(name, X_, y_):
    return sub_stratified(X_, y_, SVM_SAMPLE) if name.startswith("SVM") else (X_, y_)

def fit_data_for(name):
    return svm_sub(name, X_train, y_train)

In [ ]:
skf = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)

MLFLOW_OK = True
try:
    # local by default; set MLFLOW_TRACKING_URI (+ USERNAME/PASSWORD) to log to an online server such as DagsHub
    mlflow.set_tracking_uri(os.environ.get("MLFLOW_TRACKING_URI", "sqlite:///mlflow.db")); mlflow.set_experiment("CRC_SEER_5yr_survival")
    print("MLflow tracking ->", mlflow.get_tracking_uri())
except Exception as e:
    MLFLOW_OK = False; print("MLflow disabled:", e)

def cv_compare(names, Xc, yc, tag="main", log=True):
    """Stratified k-fold CV of every model on (Xc, yc). Returns (summary table, per-fold results)."""
    global MLFLOW_OK
    raw, rows = {}, []
    for name in names:
        t0 = time.time()
        Xn, yn = svm_sub(name, Xc, yc)
        folds = []
        for tr, va in skf.split(Xn, yn):
            m = make_model(name).fit(Xn.iloc[tr], yn.iloc[tr]); folds.append(evaluate(m, Xn.iloc[va], yn.iloc[va]))
        f = pd.DataFrame(folds); raw[name] = f
        row = {"Model": name, "n_cv": len(Xn), "time_s": round(time.time() - t0, 1)}
        for k in f.columns: row[f"{k}_mean"], row[f"{k}_std"] = f[k].mean(), f[k].std()
        rows.append(row)
        if log and MLFLOW_OK:
            try:
                with mlflow.start_run(run_name=f"cv_{tag}_{name}"):
                    mlflow.log_params({"model": name, "cv_folds": CV_FOLDS, "n_cv": len(Xn), "feature_set": FEATURE_SET, "tag": tag})
                    mlflow.log_metrics({f"cv_{k.replace(' ', '_').replace('-', '_').replace('.', '')}": row[f"{k}_mean"] for k in f.columns if not np.isnan(row[f"{k}_mean"])})
            except Exception as e:
                MLFLOW_OK = False; print("MLflow logging disabled:", e)
        print(f"[{tag}] {name:38s} AUC={row['ROC-AUC_mean']:.4f} ± {row['ROC-AUC_std']:.4f}  Brier={row['Brier_mean']:.4f}  ({row['time_s']}s)")
    return pd.DataFrame(rows).set_index("Model"), raw

all_names = [BASELINE, *TRAD, *models]
Xc, yc = sub_stratified(X_train, y_train, CV_SAMPLE)
cv_table, cv_raw = cv_compare(all_names, Xc, yc)
cv_table.to_csv(OUT / "cv_results.csv")

In [ ]:
metrics_show = ["Accuracy", "Balanced Acc.", "Precision", "Recall", "F1-score", "ROC-AUC", "PR-AUC"]
pretty = pd.DataFrame({m: cv_table.apply(lambda r: f"{r[m + '_mean']:.4f} ± {r[m + '_std']:.4f}", axis=1) for m in metrics_show})
display(pretty)

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.boxplot([cv_raw[m]["ROC-AUC"] for m in cv_raw]); ax.set_xticklabels(list(cv_raw), rotation=25, ha="right")
ax.axhline(0.5, color="red", ls="--", lw=1); ax.set_ylabel("ROC-AUC per fold"); ax.set_title(f"{CV_FOLDS}-fold stratified CV")
plt.tight_layout(); savefig("cv_roc_auc_boxplot"); plt.show()

def select_by_rule(table, candidates):
    """Selection rule (post hoc, see Step 0): max mean CV ROC-AUC; models within TIE_MARGIN are tied; tie broken by min mean CV Brier."""
    t = table.loc[candidates]
    best_auc = t["ROC-AUC_mean"].max()
    tied = t[t["ROC-AUC_mean"] >= best_auc - TIE_MARGIN]
    chosen = tied[f"{TIE_BREAK}_mean"].idxmin() if len(tied) > 1 else tied.index[0]
    return chosen, tied[["ROC-AUC_mean", "ROC-AUC_std", f"{TIE_BREAK}_mean"]]

best_name, tied = select_by_rule(cv_table, list(models))
print(f"Models tied within {TIE_MARGIN} of the best mean CV ROC-AUC:"); display(tied.round(4))
print(f"Selected by the selection rule: {best_name}")
if len(tied) > 1:
    a, b = tied.index[:2]
    d = cv_raw[a]["ROC-AUC"].values - cv_raw[b]["ROC-AUC"].values
    print(f"Fold-wise AUC difference {a} - {b}: mean {d.mean():+.4f} (range {d.min():+.4f} to {d.max():+.4f}) -> report these models as equivalent in discrimination")

## Step 10 — Hyper-parameter tuning (RandomizedSearchCV, optional Optuna)
The model that will be reported (`final_name`) is the best model of Step 9 by CV ROC-AUC – unless you set `FINAL_MODEL_OVERRIDE`
(then say in the paper that the model was chosen for explainability, not because it scored best).
Tuning uses 3-fold CV on a sample of the training set; the tuned model is then re-fitted on the **full** training set.
`tune_model()` is reused in Step 14 so that the temporal experiment is tuned only on 2010-2015 data.

In [ ]:
final_name = FINAL_MODEL_OVERRIDE or best_name
print("Final ML model:", final_name, "(override)" if FINAL_MODEL_OVERRIDE else "(selection rule)")

fs_space = {"select__estimator__C": [0.01, 0.05, 0.2]} if USE_FEATURE_SELECTION else {}
param_spaces = {
    "Logistic Regression": {"clf__C": stats.loguniform(1e-3, 1e2), **fs_space},
    "Random Forest": {"clf__n_estimators": [200, 300, 500], "clf__max_depth": [8, 12, 16, None], "clf__min_samples_leaf": [10, 30, 60],
                      "clf__max_features": ["sqrt", 0.5], **fs_space},
    "Gradient Boosting": {"clf__n_estimators": [100, 150, 250], "clf__learning_rate": [0.03, 0.05, 0.1], "clf__max_depth": [2, 3, 4],
                          "clf__subsample": [0.7, 0.85, 1.0], **fs_space},
    "XGBoost": {"clf__n_estimators": [200, 400, 700], "clf__learning_rate": [0.02, 0.05, 0.1], "clf__max_depth": [3, 4, 5, 6, 8],
                "clf__subsample": [0.7, 0.85, 1.0], "clf__colsample_bytree": [0.6, 0.8, 1.0], "clf__min_child_weight": [1, 5, 10],
                "clf__reg_lambda": [1, 5, 10], **fs_space},
    "LightGBM": {"clf__n_estimators": [200, 400, 700], "clf__learning_rate": [0.02, 0.05, 0.1], "clf__num_leaves": [15, 31, 63],
                 "clf__min_child_samples": [20, 50, 100], "clf__reg_lambda": [0, 1, 5], "clf__colsample_bytree": [0.6, 0.8, 1.0], **fs_space},
}
CLF_PARALLEL = {"Random Forest", "XGBoost", "LightGBM"}          # these already use all cores -> run search folds serially

def tune_model(name, X_, y_, tag="main"):
    """Tune `name` on (X_, y_) only. Returns (unfitted tuned pipeline, best params)."""
    base = build_pipeline(clone(models[name]))
    space = param_spaces.get(name)
    if not (DO_TUNING and space):
        print(f"[{tag}] No tuning performed for", name); return base, {}
    Xt_, yt_ = sub_stratified(*svm_sub(name, X_, y_), TUNING_SAMPLE)
    if USE_OPTUNA and HAS_OPTUNA and name == "XGBoost":
        Xo_tr, Xo_va, yo_tr, yo_va = train_test_split(Xt_, yt_, test_size=0.25, stratify=yt_, random_state=SEED)
        def objective(trial):
            p = {"clf__n_estimators": trial.suggest_int("n_estimators", 200, 700), "clf__learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
                 "clf__max_depth": trial.suggest_int("max_depth", 3, 8), "clf__subsample": trial.suggest_float("subsample", 0.6, 1.0),
                 "clf__colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0), "clf__min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
                 "clf__reg_lambda": trial.suggest_float("reg_lambda", 0.5, 10, log=True)}
            m = clone(base).set_params(**p).fit(Xo_tr, yo_tr)
            return roc_auc_score(yo_va, m.predict_proba(Xo_va)[:, 1])
        optuna.logging.set_verbosity(optuna.logging.WARNING)
        study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED)); study.optimize(objective, n_trials=OPTUNA_TRIALS)
        bp = {f"clf__{k}": v for k, v in study.best_params.items()}
        print(f"[{tag}] Optuna best validation ROC-AUC: {study.best_value:.4f}")
    else:
        search = RandomizedSearchCV(base, space, n_iter=TUNING_ITER, scoring="roc_auc", cv=StratifiedKFold(3, shuffle=True, random_state=SEED),
                                    n_jobs=1 if name in CLF_PARALLEL else -1, random_state=SEED, refit=False, verbose=1)
        search.fit(Xt_, yt_); bp = search.best_params_
        print(f"[{tag}] Best CV ROC-AUC while tuning: {search.best_score_:.4f}")
    print(f"[{tag}] Best params:", bp)
    return clone(base).set_params(**bp), bp

base_pipe = build_pipeline(clone(models[final_name]))
tuned_pipe, best_params = tune_model(final_name, X_train, y_train)

## Step 11 — Held-out test-set evaluation of every model

In [ ]:
fitted, test_rows = {}, {}
for name in all_names:
    pipe = tuned_pipe if name == final_name else make_model(name)
    Xn, yn = fit_data_for(name)
    t0 = time.time(); pipe.fit(Xn, yn); fitted[name] = pipe
    test_rows[name] = evaluate(pipe, X_test, y_test) | {"fit_time_s": round(time.time() - t0, 1)}
    print(f"{name:38s} acc={test_rows[name]['Accuracy']:.4f}  AUC={test_rows[name]['ROC-AUC']:.4f}  PR-AUC={test_rows[name]['PR-AUC']:.4f}")
test_table = pd.DataFrame(test_rows).T; test_table.to_csv(OUT / "test_results.csv")
display(test_table.round(4))

final_model = fitted[final_name]
fig, ax = plt.subplots(1, 2, figsize=(15, 5.5))
for name, m in fitted.items():
    s = get_scores(m, X_test)
    RocCurveDisplay.from_predictions(y_test, s, name=name, ax=ax[0], plot_chance_level=(name == BASELINE))
    PrecisionRecallDisplay.from_predictions(y_test, s, name=name, ax=ax[1])
ax[0].set_title("ROC curves (held-out test set)"); ax[1].set_title("Precision-Recall curves")
plt.tight_layout(); savefig("test_roc_pr_curves"); plt.show()

In [ ]:
# 95% bootstrap CIs (same resamples for every model -> paired comparison against the traditional clinical baseline)
rng = np.random.default_rng(SEED); yt_arr = y_test.values
boots = [rng.integers(0, len(yt_arr), len(yt_arr)) for _ in range(500)]
boots = [b for b in boots if len(np.unique(yt_arr[b])) == 2]
def boot_stat(s, fn=roc_auc_score): return np.array([fn(yt_arr[b], s[b]) for b in boots])

TRAD_REF = "Traditional: Stage + Age + Sex (LR)"
s_final, s_trad = get_scores(final_model, X_test), get_scores(fitted[TRAD_REF], X_test)
pred_final = predict_label(final_model, X_test)
auc_b, trad_b = boot_stat(s_final), boot_stat(s_trad)
acc_b = np.array([(yt_arr[b] == pred_final[b]).mean() for b in boots])
auc_ci, acc_ci, diff_ci = np.percentile(auc_b, [2.5, 97.5]), np.percentile(acc_b, [2.5, 97.5]), np.percentile(auc_b - trad_b, [2.5, 97.5])

print(f"{final_name}: ROC-AUC {roc_auc_score(yt_arr, s_final):.4f} (95% CI {auc_ci[0]:.4f}-{auc_ci[1]:.4f}) | accuracy {(yt_arr == pred_final).mean():.4f} (95% CI {acc_ci[0]:.4f}-{acc_ci[1]:.4f})")
print(f"{TRAD_REF}: ROC-AUC {roc_auc_score(yt_arr, s_trad):.4f}")
print(f"AUC gain over the traditional clinical model: {np.mean(auc_b - trad_b):+.4f} (95% CI {diff_ci[0]:+.4f} to {diff_ci[1]:+.4f})")
print(f"Majority-class accuracy (what 'always predict survived' scores): {max(y_test.mean(), 1 - y_test.mean()):.4f}")

### Step 11b — 95 % bootstrap confidence intervals for all key metrics, and paired model comparisons
Positive class = **survived ≥ 5 years**. So *sensitivity* = share of 5-year survivors correctly identified and *specificity* = share of patients
who died within 5 years correctly identified. Threshold = `THRESHOLD` (pre-specified). The same bootstrap resamples are used for every model,
so differences between models are **paired** comparisons. Calibration intercept (calibration-in-the-large) and slope come from logistic
recalibration of the outcome on the logit of the predicted probability (ideal: intercept 0, slope 1).

In [ ]:
def calib_int_slope(y, p):
    """Calibration-in-the-large (intercept with logit(p) as offset) and calibration slope."""
    y = np.asarray(y, float); p = np.clip(np.asarray(p, float), 1e-6, 1 - 1e-6); lp = np.log(p / (1 - p))
    a = 0.0
    for _ in range(100):                                   # Newton-Raphson for the offset-only intercept
        mu = 1 / (1 + np.exp(-(a + lp))); step = np.sum(y - mu) / max(np.sum(mu * (1 - mu)), 1e-12); a += step
        if abs(step) < 1e-10: break
    X1 = np.column_stack([np.ones_like(lp), lp]); beta = np.zeros(2)
    for _ in range(100):                                   # Newton-Raphson for logit(P(y=1)) = b0 + b1 * lp
        mu = 1 / (1 + np.exp(-X1 @ beta)); W = mu * (1 - mu)
        step = np.linalg.solve(X1.T @ (X1 * W[:, None]) + 1e-9 * np.eye(2), X1.T @ (y - mu)); beta += step
        if np.max(np.abs(step)) < 1e-10: break
    return a, beta[1]

def metric_set(y, p, thr=THRESHOLD, proba=True):
    y = np.asarray(y).astype(int); pred = (np.asarray(p) >= (thr if proba else 0.0)).astype(int)
    tp = np.sum((pred == 1) & (y == 1)); tn = np.sum((pred == 0) & (y == 0)); fp = np.sum((pred == 1) & (y == 0)); fn = np.sum((pred == 0) & (y == 1))
    out = {"ROC-AUC": roc_auc_score(y, p), "PR-AUC": average_precision_score(y, p), "Accuracy": (tp + tn) / len(y),
           "Sensitivity": tp / max(tp + fn, 1), "Specificity": tn / max(tn + fp, 1), "PPV": tp / max(tp + fp, 1), "NPV": tn / max(tn + fn, 1),
           "F1-score": 2 * tp / max(2 * tp + fp + fn, 1)}
    if proba:
        out["Brier"] = brier_score_loss(y, p); out["Calib. intercept"], out["Calib. slope"] = calib_int_slope(y, p)
    return out

def bootstrap_table(y, score_dict, n_boot=N_BOOT, seed=SEED):
    """Point estimate + percentile 95% CI for every metric and every model, on shared resamples."""
    y = np.asarray(y).astype(int); rng_ = np.random.default_rng(seed); n = len(y)
    idx = [rng_.integers(0, n, n) for _ in range(n_boot)]; idx = [b for b in idx if 0 < y[b].sum() < n]
    rows, boot_auc = [], {}
    for name, (p, proba) in score_dict.items():
        p = np.asarray(p, float); est = metric_set(y, p, proba=proba)
        bs = pd.DataFrame([metric_set(y[b], p[b], proba=proba) for b in idx]); boot_auc[name] = bs["ROC-AUC"].values
        rows.append({"Model": name, **{k: f"{v:.3f} ({bs[k].quantile(.025):.3f}-{bs[k].quantile(.975):.3f})" for k, v in est.items()}})
    return pd.DataFrame(rows).set_index("Model"), boot_auc

report_models = list(dict.fromkeys([final_name, "XGBoost", "LightGBM", "Gradient Boosting", "Logistic Regression", TRAD_REF, "Traditional: Stage only (LR)"]))
score_dict = {m: (get_scores(fitted[m], X_test), hasattr(fitted[m], "predict_proba")) for m in report_models if m in fitted}
t0 = time.time()
ci_table, boot_auc = bootstrap_table(y_test.values, score_dict)
ci_table.to_csv(OUT / "test_metrics_95ci.csv"); display(ci_table)
print(f"({len(next(iter(boot_auc.values())))} bootstrap resamples, {time.time()-t0:.0f}s)")

cmp = []
for m in report_models:
    if m == final_name or m not in boot_auc: continue
    d = boot_auc[final_name] - boot_auc[m]
    p_two = min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean()))
    cmp.append((f"{final_name} - {m}", roc_auc_score(y_test, score_dict[final_name][0]) - roc_auc_score(y_test, score_dict[m][0]),
                np.percentile(d, 2.5), np.percentile(d, 97.5), p_two))
cmp = pd.DataFrame(cmp, columns=["Comparison", "ΔAUC", "95% CI low", "95% CI high", "bootstrap p (two-sided)"]); cmp.to_csv(OUT / "paired_auc_comparisons.csv", index=False)
display(cmp.round(4))
print("Interpretation: a CI that includes 0 means the two models cannot be distinguished on this test set.")

## Step 12 — Final model: confusion matrix, calibration, subgroup performance, sanity check
* **Calibration** (does a predicted 70 % really mean 70 %?) – calibration curve + Brier score, plus an isotonic calibrator fitted on **out-of-fold** predictions of the training set (5-fold), applied to the final model, saved, and used by the web app.
* **Subgroup AUC** – the model should work across stage, sex, age and race, not only on average.
* **Sanity check** – training on *shuffled* labels must give AUC ≈ 0.5; if not, there is leakage.

In [ ]:
print(classification_report(y_test, pred_final, target_names=["Died < 5y", "Survived >= 5y"], digits=4))
has_proba = hasattr(final_model, "predict_proba")

fig, ax = plt.subplots(1, 2 if has_proba else 1, figsize=(12, 4.8), squeeze=False)
ConfusionMatrixDisplay.from_predictions(y_test, pred_final, display_labels=["Died <5y", "Survived"], cmap="Blues", ax=ax[0, 0]); ax[0, 0].set_title("Confusion matrix")
if has_proba:
    # isotonic calibrator fitted on OUT-OF-FOLD training predictions (no test data), then applied to the final model
    from sklearn.model_selection import cross_val_predict
    Xf_, yf_ = fit_data_for(final_name)
    oof = cross_val_predict(clone(tuned_pipe), Xf_, yf_, cv=StratifiedKFold(5, shuffle=True, random_state=SEED), method="predict_proba")[:, 1]
    iso = IsotonicRegression(out_of_bounds="clip", y_min=0.0, y_max=1.0).fit(oof, yf_)
    p_iso = iso.predict(s_final)
    for p_, lab_ in [(s_final, "final model"), (p_iso, "isotonic re-calibrated")]:
        fp, mp = calibration_curve(y_test, p_, n_bins=10, strategy="quantile"); ax[0, 1].plot(mp, fp, "o-", label=lab_)
    ax[0, 1].plot([0, 1], [0, 1], "k--", label="perfect"); ax[0, 1].set_xlabel("mean predicted probability"); ax[0, 1].set_ylabel("observed fraction survived")
    ax[0, 1].set_title("Calibration"); ax[0, 1].legend()
    brier_raw, brier_iso = brier_score_loss(y_test, s_final), brier_score_loss(y_test, p_iso)
    print(f"Brier score: final model {brier_raw:.4f} | after isotonic re-calibration {brier_iso:.4f} (lower = better; 0.25 = coin-flip)")
    cal_rows = []
    for lab_, p_ in [("final model (raw)", s_final), ("isotonic re-calibrated", p_iso)]:
        a_, b_ = calib_int_slope(y_test.values, p_)
        bs = [calib_int_slope(y_test.values[b], p_[b]) for b in boots[:min(len(boots), 300)]]
        cal_rows.append((lab_, a_, *np.percentile([x[0] for x in bs], [2.5, 97.5]), b_, *np.percentile([x[1] for x in bs], [2.5, 97.5]), brier_score_loss(y_test, p_)))
    cal_table = pd.DataFrame(cal_rows, columns=["Probabilities", "Calib. intercept", "int. CI low", "int. CI high", "Calib. slope", "slope CI low", "slope CI high", "Brier"])
    cal_table.to_csv(OUT / "calibration_intercept_slope.csv", index=False); display(cal_table.round(4))
plt.tight_layout(); savefig("final_confusion_calibration"); plt.show()

### Step 12b — Decision-curve analysis (clinical utility)
The decision of interest is identifying patients at **high risk of death within 5 years** (e.g. for closer follow-up). Risk of death = 1 − predicted
5-year survival probability (isotonic-calibrated for the final model). Net benefit is compared with "treat all" and "treat none" across threshold
probabilities. A model is useful only in the threshold range where its curve lies above both reference lines.

In [ ]:
def net_benefit(event, risk, pt):
    n = len(event); treat = risk >= pt
    return np.sum(treat & (event == 1)) / n - np.sum(treat & (event == 0)) / n * pt / (1 - pt)

event = (1 - y_test.values).astype(int)                      # 1 = died within 5 years
pts = np.round(np.arange(0.05, 0.96, 0.01), 2)
dca_scores = {f"{final_name} (calibrated)": 1 - (p_iso if has_proba else s_final)}
for m in ["Logistic Regression", TRAD_REF]:
    if m in fitted and hasattr(fitted[m], "predict_proba"): dca_scores[m] = 1 - fitted[m].predict_proba(X_test)[:, 1]
dca = pd.DataFrame({"threshold": pts, **{k: [net_benefit(event, r, pt) for pt in pts] for k, r in dca_scores.items()},
                    "Treat all": [event.mean() - (1 - event.mean()) * pt / (1 - pt) for pt in pts], "Treat none": 0.0})
dca.to_csv(OUT / "decision_curve.csv", index=False)
plt.figure(figsize=(8, 5))
for c in dca.columns[1:]:
    plt.plot(dca.threshold, dca[c], lw=2 if c in dca_scores else 1, ls="-" if c in dca_scores else "--", label=c)
plt.ylim(-0.05, max(0.05, event.mean()) + 0.05); plt.xlabel("threshold probability of death within 5 years"); plt.ylabel("net benefit")
plt.title("Decision-curve analysis (held-out test set)"); plt.legend(fontsize=8); plt.tight_layout(); savefig("decision_curve"); plt.show()
best_rng = dca.loc[dca[f"{final_name} (calibrated)"] > dca[["Treat all", "Treat none"]].max(axis=1), "threshold"]
print(f"{final_name} has higher net benefit than treat-all/none for thresholds {best_rng.min():.2f}-{best_rng.max():.2f}" if len(best_rng) else "No threshold range with added net benefit.")

In [ ]:
sub = test_b.copy(); sub["score"] = s_final
sub["Age band"] = pd.cut(sub["Age_years"], [0, 50, 65, 80, 200], labels=["<50", "50-64", "65-79", "80+"])
rows = []
for g in ["Stage_Simple", "Sex", "Age band", "Race", "Site"]:
    for lvl, part in sub.groupby(g, observed=True):
        if len(part) >= 300 and part["y5"].nunique() == 2:
            rows.append((g, str(lvl), len(part), roc_auc_score(part["y5"], part["score"])))
sg = pd.DataFrame(rows, columns=["Subgroup", "Level", "n", "ROC-AUC"]); sg.to_csv(OUT / "subgroup_auc.csv", index=False)
display(sg.round(3))
plt.figure(figsize=(7, 6)); plt.barh([f"{a}: {b}" for a, b in zip(sg.Subgroup, sg.Level)][::-1], sg["ROC-AUC"][::-1], color="#2a9d8f")
plt.axvline(roc_auc_score(y_test, s_final), color="k", ls="--", lw=1); plt.xlim(0.5, 1); plt.xlabel("ROC-AUC"); plt.title("Performance by subgroup (dashed = overall)")
plt.tight_layout(); savefig("subgroup_auc"); plt.show()

In [ ]:
Xtr_f, ytr_f = fit_data_for(final_name)
shuf = []
for i in range(2):
    y_shuf = pd.Series(np.random.default_rng(SEED + i).permutation(ytr_f.values), index=ytr_f.index)
    m = clone(tuned_pipe).fit(Xtr_f, y_shuf); shuf.append(roc_auc_score(y_test, get_scores(m, X_test)))
print(f"Real labels: ROC-AUC {roc_auc_score(y_test, s_final):.4f} | shuffled labels: {np.round(shuf, 4)} (should be ~0.50)")
assert max(shuf) < 0.55, "Shuffled-label AUC is well above 0.5 -> check for leakage!"
print("Sanity check passed: no sign of leakage.")

## Step 13 — Sensitivity analysis: which variables are really available at prediction time?
The same (tuned) classifier is trained on feature sets **A** (diagnosis & staging), **B** (+ pathology) and **C** (+ treatment).
If performance jumps a lot from A → C, the gain comes from information that is only known *after* treatment decisions, which weakens the claim of
"prediction at diagnosis". Report the set you use and show this table.

**Prediction time point used in the paper:** the primary model (Set B) is a **post-surgical pathological-assessment** model – lymph-node counts, node ratio, perineural invasion and tumour deposits are only known after resection. Set A is reported in full as the **diagnosis-time** model. The cross-table below shows how strongly "0 nodes examined" encodes "no surgery", i.e. why Set B already carries treatment information.

In [ ]:
clf_t = clone(final_model.named_steps["clf"])
sens, sens_models, sens_scores = {}, {}, {}
for fs, label in [("A", "A: diagnosis + staging"), ("B", "B: + pathology"), ("C", "C: + treatment")]:
    p = build_pipeline(clone(clf_t), fs)
    Xn, yn = fit_data_for(final_name); p.fit(Xn, yn); sens[label] = evaluate(p, X_test, y_test)
    sens_models[fs] = p; sens_scores[f"Set {label}"] = (get_scores(p, X_test), True)
sens = pd.DataFrame(sens).T[["Accuracy", "Balanced Acc.", "F1-score", "ROC-AUC", "PR-AUC", "Brier"]]; sens.to_csv(OUT / "sensitivity_feature_sets.csv")
display(sens.round(4))
sens_ci, _ = bootstrap_table(y_test.values, sens_scores, n_boot=min(N_BOOT, 500))
sens_ci.to_csv(OUT / "sensitivity_feature_sets_95ci.csv"); display(sens_ci[["ROC-AUC", "PR-AUC", "Brier", "Calib. slope"]])
ax = sens[["ROC-AUC", "PR-AUC", "Balanced Acc."]].plot(kind="bar", figsize=(8, 4), rot=15); ax.set_ylim(0.5, 1); ax.set_title("Sensitivity analysis: feature sets")
plt.tight_layout(); savefig("sensitivity_feature_sets"); plt.show()

# how much treatment information is hidden in the "pathology" variables of Set B?
ne0 = np.where(C["Nodes_Examined"].isna(), "nodes examined: missing", np.where(C["Nodes_Examined"].eq(0), "nodes examined: 0", "nodes examined: >0"))
leak = pd.crosstab(pd.Series(ne0, index=C.index, name="Set B variable"), C["Surgery"], normalize="index").round(3)
leak.to_csv(OUT / "nodes_examined_vs_surgery.csv"); print("Row % of surgery status by nodes-examined category:"); display(leak)
diag_model = sens_models["A"]                                  # saved in Step 17 as the diagnosis-time model

## Step 14 — Temporal validation (+ optional external validation)
**Strict temporal validation:** the 2010-2015 patients form a separate development cohort. **Model selection (same rule),
preprocessing, feature selection and hyper-parameter tuning are all performed only inside 2010-2015.** The selected model is refitted on all
2010-2015 patients and evaluated once on 2016-2018 (diagnosed later: different coding editions, newer treatments).
Report: *"For temporal validation, all preprocessing, feature selection and hyperparameter tuning were performed exclusively within the
2010–2015 development cohort, and the 2016–2018 cohort was held out until final evaluation."*

**External validation:** see the separate notebook `CRC_GENIE_External_Validation.ipynb` (AACR Project GENIE BPC CRC).

In [ ]:
lab_all = C[C.y5.notna()]
dev, val = lab_all[lab_all.Year <= TEMPORAL_CUT], lab_all[lab_all.Year > TEMPORAL_CUT]
Xd, yd = dev[ALLX], dev.y5.astype(int)
Xv_t, yv_t = val[ALLX], val.y5.astype(int)
print(f"Development {dev.Year.min()}-{dev.Year.max()}: n={len(dev):,} | temporal test {val.Year.min()}-{val.Year.max()}: n={len(val):,} (never used before this evaluation)")

# 1) model selection inside 2010-2015 only (same candidates, same rule)
Xdc, ydc = sub_stratified(Xd, yd, CV_SAMPLE)
cv_t, cv_t_raw = cv_compare(list(models), Xdc, ydc, tag="temporal-dev", log=False)
cv_t.to_csv(OUT / "temporal_dev_cv_results.csv")
t_name, t_tied = select_by_rule(cv_t, list(models))
print("Temporal development cohort -> selected:", t_name); display(t_tied.round(4))

# 2) tuning inside 2010-2015 only, 3) refit on all 2010-2015, 4) one evaluation on 2016-2018
t_pipe, t_params = tune_model(t_name, Xd, yd, tag="temporal-dev")
Xfit, yfit = svm_sub(t_name, Xd, yd)
m_t = t_pipe.fit(Xfit, yfit)
temporal = evaluate(m_t, Xv_t, yv_t)
display(pd.Series(temporal).round(4).to_frame("strict temporal validation").T)
t_ci, _ = bootstrap_table(yv_t.values, {f"{t_name} (dev 2010-{TEMPORAL_CUT})": (get_scores(m_t, Xv_t), hasattr(m_t, "predict_proba"))}, n_boot=min(N_BOOT, 500))
t_ci.to_csv(OUT / "temporal_validation_95ci.csv"); display(t_ci)
print("Random-split test ROC-AUC for comparison:", round(test_table.loc[final_name, "ROC-AUC"], 4))
print("Note: 2016-2018 patients alive with < 60 months follow-up are censored and excluded from the binary evaluation - report this count.")
print(f"Censored (alive, <60 mo) among 2016-{val.Year.max()} diagnoses: {int(C[(C.Year > TEMPORAL_CUT)].y5.isna().sum()):,}")

## Step 15 — Survival models, censoring and IPCW
The binary 5-year label throws away *when* patients died and the censored patients. Survival models use the whole follow-up:
* **Kaplan-Meier** curves + log-rank test (non-parametric)
* **Cox proportional-hazards** (the traditional statistical method – hazard ratios are interpretable)
* **XGBoost-Cox** (gradient-boosted survival model; 5-year survival obtained with a Breslow baseline)
* optional **Random Survival Forest, Gradient-Boosting Survival, CoxNet** (`scikit-survival`)
* **IPCW** (inverse-probability-of-censoring weighting) so that the Brier score and a re-weighted classifier correct for censored patients.

Common yardsticks: **C-index**, **5-year AUC** (labelled patients), **time-dependent AUC at 60 months** (needs scikit-survival), **IPCW Brier score at 60 months**.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4.8))
kmf = KaplanMeierFitter(); km5 = {}
for grp in ["Localized", "Regional", "Distant"]:
    m = C.Stage_Simple == grp
    kmf.fit(C.Time[m], C.Event[m], label=f"{grp} (n={int(m.sum()):,})"); kmf.plot_survival_function(ax=ax[0]); km5[grp] = float(kmf.predict(HORIZON))
ax[0].axvline(HORIZON, ls=":", c="k"); ax[0].set_title("Kaplan-Meier overall survival by SEER summary stage"); ax[0].set_xlabel("months since diagnosis"); ax[0].set_ylim(0, 1)
C["Age band"] = pd.cut(C["Age_years"], [0, 50, 65, 80, 200], labels=["<50", "50-64", "65-79", "80+"])
for grp, m in C.groupby("Age band", observed=True).groups.items():
    kmf.fit(C.loc[m, "Time"], C.loc[m, "Event"], label=str(grp)); kmf.plot_survival_function(ax=ax[1], ci_show=False)
ax[1].axvline(HORIZON, ls=":", c="k"); ax[1].set_title("Kaplan-Meier by age band"); ax[1].set_xlabel("months since diagnosis"); ax[1].set_ylim(0, 1)
plt.tight_layout(); savefig("km_curves"); plt.show()

k = C[C.Stage_Simple.isin(["Localized", "Regional", "Distant"])]
lr = multivariate_logrank_test(k.Time, k.Stage_Simple, k.Event)
print("Kaplan-Meier 5-year survival by stage:", {g: round(v, 3) for g, v in km5.items()}, f"| log-rank p = {lr.p_value:.3g}")

In [ ]:
# ---------- design matrices (same preprocessing as the classifiers, fitted on the training set) ----------
tr_s = train_all.sample(min(SURV_SAMPLE, len(train_all)), random_state=SEED)
test_s = test_all.sample(min(20000, len(test_all)), random_state=SEED)        # common test sample for the survival comparison
prep_s = make_preprocessor(NUM_COLS, CAT_COLS).fit(tr_s)
snames = list(prep_s.get_feature_names_out())
Xs_tr = pd.DataFrame(prep_s.transform(tr_s), columns=snames, index=tr_s.index)
Xs_te = pd.DataFrame(prep_s.transform(test_s), columns=snames, index=test_s.index)
keep_cols = [c for c in snames if Xs_tr[c].std() > 1e-8]; Xs_tr, Xs_te = Xs_tr[keep_cols], Xs_te[keep_cols]

# ---------- IPCW machinery ----------
km_cens = KaplanMeierFitter().fit(train_all.Time, 1 - train_all.Event)                  # censoring distribution G(t)
G = lambda t: np.clip(km_cens.survival_function_at_times(np.atleast_1d(t)).values, 0.05, 1.0)

def ipcw_brier(time, event, S, horizon=HORIZON):
    t, e, S = np.asarray(time, float), np.asarray(event).astype(int), np.asarray(S, float)
    died = (t < horizon) & (e == 1); alive = t >= horizon
    w = np.zeros(len(t)); w[died] = 1 / G(t[died]); w[alive] = 1 / G(np.full(int(alive.sum()), horizon))
    return float(np.sum(w * (alive.astype(float) - S) ** 2) / len(t))

def breslow(time, event, lp):
    o = np.argsort(time); t, e, r = np.asarray(time)[o], np.asarray(event)[o], np.exp(np.asarray(lp)[o])
    return t, np.cumsum(e / np.cumsum(r[::-1])[::-1])
def surv_from_lp(grid, H0, lp, t=HORIZON):
    i = np.searchsorted(grid, t, side="right") - 1
    return np.exp(-(H0[i] if i >= 0 else 0.0) * np.exp(lp))

try:
    from sksurv.metrics import cumulative_dynamic_auc
    y_tr_surv = Surv.from_arrays(train_all.Event.astype(bool).values, train_all.Time.values)
    def td_auc(S60):
        m = (test_s.Time.values <= train_all.Time.max()) & (test_s.Time.values >= train_all.Time[train_all.Event == 1].min())
        y_te = Surv.from_arrays(test_s.Event.astype(bool).values[m], test_s.Time.values[m])
        return float(cumulative_dynamic_auc(y_tr_surv, y_te, 1 - np.asarray(S60)[m], [HORIZON])[0][0])
except Exception:
    td_auc = lambda S60: np.nan

lab_mask = test_s.y5.notna().values
def surv_report(S60):
    S60 = np.asarray(S60, float)
    return {"C-index": concordance_index(test_s.Time, S60, test_s.Event), "5-y AUC (labelled)": roc_auc_score(test_s.y5[lab_mask], S60[lab_mask]),
            "tdAUC(60m)": td_auc(S60), "IPCW Brier(60m)": ipcw_brier(test_s.Time, test_s.Event, S60)}
surv_rows = {}

In [ ]:
# ---------- Cox proportional hazards (traditional statistical model) ----------
t0 = time.time()
cox_df = Xs_tr.assign(Time=tr_s.Time.values, Event=tr_s.Event.values)
cph = CoxPHFitter(penalizer=0.1, l1_ratio=0.0).fit(cox_df, duration_col="Time", event_col="Event")
S60_cox = cph.predict_survival_function(Xs_te, times=[HORIZON]).iloc[0].values
surv_rows["Cox PH (lifelines)"] = surv_report(S60_cox); print(f"Cox fitted in {time.time()-t0:.0f}s")

hr = cph.summary[["exp(coef)", "exp(coef) lower 95%", "exp(coef) upper 95%", "p"]].rename(columns={"exp(coef)": "HR"})
hr["feature"] = hr.index; hr["variable"] = [origin_of(n) for n in hr.index]
hr.sort_values("p").to_csv(OUT / "cox_hazard_ratios.csv")
top_hr = hr.reindex(hr["HR"].sub(1).abs().sort_values(ascending=False).index).head(15)[::-1]
plt.figure(figsize=(8, 6)); plt.errorbar(top_hr["HR"], range(len(top_hr)), xerr=[top_hr["HR"] - top_hr["exp(coef) lower 95%"], top_hr["exp(coef) upper 95%"] - top_hr["HR"]], fmt="o", color="#1d3557")
plt.yticks(range(len(top_hr)), top_hr["feature"], fontsize=8); plt.axvline(1, color="red", ls="--"); plt.xlabel("hazard ratio (95% CI)"); plt.title("Cox model: largest hazard ratios")
plt.tight_layout(); savefig("cox_hazard_ratios"); plt.show()
print("Note: hazard ratios are for standardised numeric columns and one-hot categories (penalised Cox). Check the proportional-hazards assumption (cph.check_assumptions) on a sample before interpreting HRs in the paper.")

# ---------- XGBoost survival (Cox objective) + Breslow baseline ----------
t0 = time.time()
y_cox = np.where(tr_s.Event.values == 1, tr_s.Time.values, -tr_s.Time.values)
xgb_cox = XGBRegressor(objective="survival:cox", n_estimators=300, learning_rate=0.05, max_depth=4, subsample=0.8, colsample_bytree=0.8,
                       tree_method="hist", n_jobs=-1, random_state=SEED).fit(Xs_tr, y_cox)
grid, H0 = breslow(tr_s.Time.values, tr_s.Event.values, xgb_cox.predict(Xs_tr, output_margin=True))
S60_xgbcox = surv_from_lp(grid, H0, xgb_cox.predict(Xs_te, output_margin=True))
surv_rows["XGBoost-Cox"] = surv_report(S60_xgbcox); print(f"XGBoost-Cox fitted in {time.time()-t0:.0f}s")

In [ ]:
# ---------- OPTIONAL: scikit-survival models (Random Survival Forest, Gradient-Boosting Survival, CoxNet) ----------
if RUN_SKSURV and HAS_SKSURV:
    from sksurv.ensemble import RandomSurvivalForest, GradientBoostingSurvivalAnalysis
    from sksurv.linear_model import CoxnetSurvivalAnalysis

    def sk_S60(model, X, chunk=2500):
        out = []
        for i in range(0, len(X), chunk):
            fns = model.predict_survival_function(X.iloc[i:i + chunk])
            out.extend([f(min(HORIZON, f.domain[1])) for f in fns])
        return np.array(out)

    n_sk = min(20000, len(Xs_tr)); sk_idx = Xs_tr.sample(n_sk, random_state=SEED).index
    y_sk = Surv.from_arrays(tr_s.loc[sk_idx, "Event"].astype(bool).values, tr_s.loc[sk_idx, "Time"].values)
    sk_models = {"Random Survival Forest": RandomSurvivalForest(n_estimators=100, min_samples_leaf=50, max_features="sqrt", n_jobs=-1, random_state=SEED),
                 "Gradient-Boosting Survival": GradientBoostingSurvivalAnalysis(n_estimators=150, learning_rate=0.1, max_depth=3, subsample=0.8, random_state=SEED),
                 "CoxNet (elastic net)": CoxnetSurvivalAnalysis(l1_ratio=0.5, alpha_min_ratio=0.01, fit_baseline_model=True)}
    for nm, mdl in sk_models.items():
        t0 = time.time()
        try:
            mdl.fit(Xs_tr.loc[sk_idx], y_sk); surv_rows[nm] = surv_report(sk_S60(mdl, Xs_te)); print(f"{nm} done in {time.time()-t0:.0f}s")
        except Exception as ex:
            print(f"{nm} skipped: {ex}")
else:
    print("scikit-survival not installed (or RUN_SKSURV=False) -> optional survival models skipped.  pip install scikit-survival")

In [ ]:
# ---------- classifiers evaluated with the same survival yardsticks (their output is S(60) = P(survive 5 years)) ----------
for nm in [final_name, "Logistic Regression", TRAD_REF]:
    if hasattr(fitted[nm], "predict_proba"):
        surv_rows[f"Classifier: {nm}"] = surv_report(fitted[nm].predict_proba(test_s[ALLX])[:, 1])

# ---------- IPCW-weighted classifier ----------
Xn, yn = fit_data_for(final_name)
w = pd.Series(np.where(train_b.Time >= HORIZON, 1 / G(HORIZON)[0], 1 / G(train_b.Time.values)), index=train_b.index).clip(upper=10)
m_ipcw = clone(tuned_pipe).fit(Xn, yn, clf__sample_weight=w.loc[Xn.index].values)
if hasattr(m_ipcw, "predict_proba"):
    surv_rows[f"Classifier (IPCW-weighted): {final_name}"] = surv_report(m_ipcw.predict_proba(test_s[ALLX])[:, 1])
print(f"Censoring: {100*C.y5.isna().mean():.1f}% of the cohort is censored before 60 months; IPCW weights range {w.min():.2f}-{w.max():.2f}")

surv_table = pd.DataFrame(surv_rows).T; surv_table.to_csv(OUT / "survival_model_comparison.csv")
print(f"\nSurvival comparison on a common test sample of {len(test_s):,} patients (higher C-index / AUC and lower Brier are better):")
display(surv_table.round(4))

## Step 16 — Explainable AI: SHAP (global + patient-level), LIME, partial-dependence plots
SHAP values are summed back from one-hot columns to the original clinical variable.
**PDP** shows the *average* effect of one variable on the predicted 5-year survival.

In [ ]:
prep_only = Pipeline(final_model.steps[:-1]); clf_final = final_model.named_steps["clf"]
sel_names = selected_names(final_model)
feature_origin = {n: origin_of(n) for n in sel_names}

Xs_raw = X_test.sample(min(2000, len(X_test)), random_state=SEED)
Xs = prep_only.transform(Xs_raw)
Xbg = prep_only.transform(X_train.sample(min(500, len(X_train)), random_state=SEED))

if isinstance(clf_final, (RandomForestClassifier, GradientBoostingClassifier, XGBClassifier, LGBMClassifier)):
    expl = shap.TreeExplainer(clf_final); sv = expl.shap_values(Xs)
    if isinstance(sv, list): sv = sv[1]
    sv = np.asarray(sv); sv = sv[:, :, 1] if sv.ndim == 3 else sv
elif isinstance(clf_final, LogisticRegression):
    sv = np.asarray(shap.LinearExplainer(clf_final, Xbg).shap_values(Xs))
else:                                                       # SVM -> slow model-agnostic explainer on a small sample
    Xs_raw, Xs = Xs_raw.iloc[:100], Xs[:100]
    sv = np.asarray(shap.KernelExplainer(clf_final.decision_function, shap.sample(Xbg, 50)).shap_values(Xs, nsamples=100))
print("SHAP matrix:", sv.shape)

plt.figure(); shap.summary_plot(sv, Xs, feature_names=sel_names, show=False, max_display=15)
plt.title("SHAP summary"); plt.tight_layout(); savefig("shap_beeswarm"); plt.show()

mean_abs = pd.Series(np.abs(sv).mean(0), index=sel_names)
shap_by_feature = mean_abs.groupby(feature_origin).sum().sort_values(ascending=False)
plt.figure(figsize=(7, 6)); shap_by_feature.head(15)[::-1].plot(kind="barh", color="#1d3557")
plt.xlabel("mean |SHAP| (summed over encoded columns)"); plt.title("Global importance (SHAP) by clinical variable")
plt.tight_layout(); savefig("shap_global_importance"); plt.show()
display(shap_by_feature.head(10).round(5).to_frame("mean_abs_shap"))

In [ ]:
# ---- patient-level SHAP (first patient of the sample) ----
i = 0; patient = Xs_raw.iloc[[i]][FEATS]
display(patient.T.rename(columns={patient.index[0]: "patient value"}))
contrib = pd.Series(sv[i], index=sel_names).groupby(feature_origin).sum()
top = contrib.reindex(contrib.abs().sort_values(ascending=False).head(10).index)[::-1]
plt.figure(figsize=(7, 4.5)); top.plot(kind="barh", color=["#2a9d8f" if v > 0 else "#e76f51" for v in top])
plt.axvline(0, color="k", lw=.8); plt.xlabel("SHAP value (green: towards survival, red: against)"); plt.title(f"Patient-level explanation ({final_name})")
plt.tight_layout(); savefig("shap_patient"); plt.show()

# ---- LIME (same patient) ----
if hasattr(clf_final, "predict_proba"):
    Xtr_lime = prep_only.transform(X_train.sample(min(5000, len(X_train)), random_state=SEED))
    lime_expl = LimeTabularExplainer(Xtr_lime, feature_names=sel_names, class_names=["Died <5y", "Survived"], mode="classification", discretize_continuous=True, random_state=SEED)
    exp = lime_expl.explain_instance(Xs[i], clf_final.predict_proba, num_features=10, num_samples=3000)
    exp.as_pyplot_figure(); plt.tight_layout(); savefig("lime_patient"); plt.show()
    display(pd.DataFrame(exp.as_list(), columns=["condition", "weight"]))
else:
    print("Final model has no predict_proba (SVM) -> LIME skipped.")

In [ ]:
# ---- Partial-dependence plots (average effect on predicted 5-year survival) ----
pdp_feats = [f for f in ["Age_years", "Tumor_Size_mm", "Nodes_Positive", "Node_Ratio"] if f in NUM_COLS]
Xp = X_test.sample(min(5000, len(X_test)), random_state=SEED).copy()
for f in pdp_feats: Xp[f] = Xp[f].fillna(Xp[f].median())
disp = PartialDependenceDisplay.from_estimator(final_model, Xp, pdp_feats, grid_resolution=20, kind="average", n_cols=len(pdp_feats))
disp.figure_.set_size_inches(4.2 * len(pdp_feats), 3.8); disp.figure_.suptitle(f"Partial dependence ({final_name})", y=1.02)
plt.tight_layout(); savefig("pdp"); plt.show()

## Step 17 — Save model, metadata and log the final run (MLflow)

In [ ]:
joblib.dump(final_model, ART / "crc_model.joblib")                 # primary model (Set B, post-surgical)
if has_proba: joblib.dump(iso, ART / "crc_calibrator.joblib")      # out-of-fold isotonic calibrator for crc_model
joblib.dump(diag_model, ART / "crc_model_setA_diagnosis.joblib")    # diagnosis-time model (Set A), same classifier settings

# background for the app's KernelExplainer (only used if the final model is an SVM): 10 k-means CENTRES, not patient rows
centers = KMeans(10, n_init=3, random_state=SEED).fit(prep_only.transform(X_train.sample(min(2000, len(X_train)), random_state=SEED))).cluster_centers_
np.save(ART / "background.npy", centers)

cat_options = {c: sorted(C[c].dropna().astype(str).unique().tolist()) for c in CAT_COLS}
meta = {
    "project": "Explainable AI-driven 5-year survival prediction in colorectal cancer (SEER)",
    "model_name": final_name, "feature_set": FEATURE_SET, "prediction_timepoint": TIMEPOINT[FEATURE_SET],
    "target": "y5 = 1 if SEER survival months >= 60 (alive or dead at >= 60 mo); 0 if died (all causes) < 60 mo; alive < 60 mo = censored, excluded",
    "positive_class": "survived >= 5 years", "threshold": THRESHOLD,
    "threshold_applies_to": "uncalibrated predict_proba()[:, 1] of crc_model.joblib",
    "calibrator": "crc_calibrator.joblib (isotonic, out-of-fold on training set); apply to the uncalibrated probability" if has_proba else None,
    "selection_rule": f"max mean {CV_FOLDS}-fold CV ROC-AUC; ties within {TIE_MARGIN} broken by min mean CV {TIE_BREAK}",
    "setA_model": {"file": "crc_model_setA_diagnosis.joblib", "feature_cols": feats_for("A")[0] + feats_for("A")[1]},
    "age_coding": "SEER 5-year age band converted to the band midpoint (90+ -> 92)",
    "cohort": "SEER 17 (Nov 2025 sub), colon & rectum, dx 2010-2018, first/only primary, age >= 20, not DCO/autopsy, not in situ",
    "feature_cols": FEATS, "numeric": {c: {"min": float(C[c].min()), "max": float(C[c].max()), "median": float(C[c].median())} for c in NUM_COLS},
    "categorical": cat_options, "selected_feature_names": sel_names, "feature_origin": feature_origin,
    "test_metrics": {k: (None if pd.isna(v) else float(v)) for k, v in test_table.loc[final_name].items()},
    "auc_95ci": [float(auc_ci[0]), float(auc_ci[1])], "n_train": int(len(X_train)), "n_test": int(len(X_test)),
    "best_params": {k: (v if isinstance(v, (int, float, str, type(None))) else str(v)) for k, v in best_params.items()},
    "trained_on": datetime.now().isoformat(timespec="seconds"),
    "versions": {"python": platform.python_version(), "sklearn": __import__("sklearn").__version__, "pandas": pd.__version__},
}
(ART / "model_meta.json").write_text(json.dumps(meta, indent=2, default=str))
print("Saved:", [p.name for p in ART.iterdir()])

if MLFLOW_OK:
    try:
        with mlflow.start_run(run_name=f"FINAL_{final_name}"):
            mlflow.log_params({"model": final_name, "feature_set": FEATURE_SET, **{k: str(v)[:200] for k, v in best_params.items()}})
            mlflow.log_metrics({f"test_{k.replace(' ', '_').replace('-', '_').replace('.', '')}": v for k, v in meta["test_metrics"].items() if v is not None})
            mlflow.log_artifact(str(ART / "crc_model.joblib")); mlflow.log_artifact(str(ART / "model_meta.json")); mlflow.log_artifacts(str(FIG), artifact_path="figures")
        print("Logged to MLflow -> view with:  mlflow ui --backend-store-uri sqlite:///mlflow.db --port 5001")
    except Exception as e:
        print("MLflow final logging skipped:", e)

### Step 17b — Transportable core model for external validation (trained on SEER only)
GENIE BPC CRC has no marital status, income, rurality, tumour size or lymph-node counts, so the Set A/B models cannot be applied there without
inventing values. This **core model** uses only predictors that exist in both datasets, is trained **only on the SEER training set** with the same
classifier settings as the final model, gets its own out-of-fold isotonic calibrator, and is frozen. It is reported in the paper as a separate,
reduced model, with its SEER test performance shown next to the full model.

In [ ]:
CORE_NUM = ["Age_years"]
CORE_CAT = ["Sex", "Race", "Site", "Histology", "Grade", "Summary_Stage", "Stage_Group", "T_Stage", "N_Stage", "M_Stage", "CEA"]
core_pipe = Pipeline([("prep", make_preprocessor(CORE_NUM, CORE_CAT)), ("select", make_selector()), ("clf", clone(final_model.named_steps["clf"]))])
Xcore, ycore = fit_data_for(final_name)
core_model = clone(core_pipe).fit(Xcore, ycore)
from sklearn.model_selection import cross_val_predict
oof_core = cross_val_predict(clone(core_pipe), Xcore, ycore, cv=StratifiedKFold(5, shuffle=True, random_state=SEED), method="predict_proba")[:, 1]
iso_core = IsotonicRegression(out_of_bounds="clip", y_min=0.0, y_max=1.0).fit(oof_core, ycore)
core_ci, _ = bootstrap_table(y_test.values, {"Core (transportable)": (core_model.predict_proba(X_test)[:, 1], True),
                                             f"{final_name} (Set {FEATURE_SET})": (s_final, True)}, n_boot=min(N_BOOT, 500))
core_ci.to_csv(OUT / "core_model_seer_test_95ci.csv"); display(core_ci[["ROC-AUC", "PR-AUC", "Brier", "Calib. slope"]])
joblib.dump(core_model, ART / "crc_model_core_transportable.joblib"); joblib.dump(iso_core, ART / "crc_calibrator_core.joblib")
meta["core_model"] = {"file": "crc_model_core_transportable.joblib", "calibrator": "crc_calibrator_core.joblib", "feature_cols": CORE_NUM + CORE_CAT,
                      "seer_test_auc": float(roc_auc_score(y_test, core_model.predict_proba(X_test)[:, 1]))}
(ART / "model_meta.json").write_text(json.dumps(meta, indent=2, default=str))
print("Saved core model; SEER test ROC-AUC", round(meta["core_model"]["seer_test_auc"], 4))

## Step 18 — Deployment: Flask web app + Docker
The next cell writes `app.py`. The input form is built from `artifacts/model_meta.json`; numeric fields may be left blank (the pipeline imputes them),
categorical fields default to *Unknown*. The app returns the 5-year survival probability and the top-5 patient-level SHAP contributions, and shows the
clinical-prototype disclaimer. Run it with `python app.py`.

In [ ]:
%%writefile app.py
"""CRC 5-year survival (SEER) - clinical decision-support PROTOTYPE (not a diagnostic device)."""
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from flask import Flask, jsonify, request, render_template_string
from sklearn.pipeline import Pipeline

BASE = Path(__file__).resolve().parent
ART = BASE / "artifacts"
pipe = joblib.load(ART / "crc_model.joblib")
meta = json.loads((ART / "model_meta.json").read_text())

FEATURES, NUMERIC, CATEGORICAL = meta["feature_cols"], meta["numeric"], meta["categorical"]
NAMES, ORIGIN = meta["selected_feature_names"], meta["feature_origin"]
prep = Pipeline(pipe.steps[:-1])
CAL = joblib.load(ART / "crc_calibrator.joblib") if (ART / "crc_calibrator.joblib").exists() else None
THRESHOLD = meta.get("threshold", 0.5)
clf = pipe.steps[-1][1]

DISCLAIMER = ("Research prototype for clinical decision support, trained on SEER registry data. Not a diagnostic device; "
              "it must not replace clinical judgement.")

explainer = None
try:
    import shap
    if clf.__class__.__name__ in ("RandomForestClassifier", "GradientBoostingClassifier", "XGBClassifier", "LGBMClassifier"):
        explainer = shap.TreeExplainer(clf)
except Exception:
    explainer = None

# SVM has neither trees nor coefficients -> model-agnostic KernelExplainer (a few seconds per patient)
kernel_explainer = None
BG = ART / "background.npy"
if explainer is None and not hasattr(clf, "coef_") and BG.exists():
    try:
        import shap
        _f = clf.decision_function if hasattr(clf, "decision_function") else (lambda a: clf.predict_proba(a)[:, 1])
        kernel_explainer = shap.KernelExplainer(_f, np.load(BG))
    except Exception:
        kernel_explainer = None


def explain(xrow):
    """Top-5 patient-level contributions, summed back to the original clinical variable."""
    if explainer is not None:
        sv = explainer.shap_values(xrow)
        if isinstance(sv, list):
            sv = sv[1]
        sv = np.asarray(sv)
        contrib = (sv[:, :, 1] if sv.ndim == 3 else sv)[0]
    elif hasattr(clf, "coef_"):
        contrib = clf.coef_[0] * xrow[0]
    elif kernel_explainer is not None:
        sv = kernel_explainer.shap_values(xrow, nsamples=200, silent=True)
        contrib = np.asarray(sv[0] if isinstance(sv, list) else sv).reshape(-1)
    else:
        return []
    agg = {}
    for n, v in zip(NAMES, contrib):
        agg[ORIGIN[n]] = agg.get(ORIGIN[n], 0.0) + float(v)
    top = sorted(agg.items(), key=lambda kv: abs(kv[1]), reverse=True)[:5]
    return [{"feature": k, "impact": round(v, 4),
             "direction": "towards survival" if v > 0 else "towards non-survival"} for k, v in top]


def predict_one(payload):
    row = {}
    for f in FEATURES:
        v = payload.get(f)
        if f in NUMERIC:
            row[f] = float(v) if v not in (None, "") else np.nan
        else:
            row[f] = str(v) if v not in (None, "") else "Unknown"
    Xt = prep.transform(pd.DataFrame([row], columns=FEATURES))
    if hasattr(clf, "predict_proba"):
        p_raw = float(clf.predict_proba(Xt)[0, 1])
    else:
        p_raw = float(1.0 / (1.0 + np.exp(-clf.decision_function(Xt)[0])))
    p = float(CAL.predict([p_raw])[0]) if CAL is not None else p_raw          # calibrated probability shown to the user
    return {"model": meta["model_name"], "survival_probability": round(p, 4), "uncalibrated_probability": round(p_raw, 4),
            "prediction": ("Predicted 5-year survival at or above the model threshold" if p_raw >= THRESHOLD
                           else "Predicted 5-year survival below the model threshold (higher risk)"),
            "top_factors": explain(Xt), "disclaimer": DISCLAIMER}


PAGE = """
<!doctype html><html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width, initial-scale=1">
<title>CRC 5-year survival (SEER prototype)</title>
<style>
 body{font-family:system-ui,Arial,sans-serif;max-width:980px;margin:24px auto;padding:0 16px;color:#1b1b1b}
 .warn{background:#fff4e5;border-left:5px solid #f59e0b;padding:10px 14px;margin:12px 0;font-size:.92rem}
 form{display:grid;grid-template-columns:repeat(auto-fill,minmax(220px,1fr));gap:12px}
 label{font-size:.82rem;font-weight:600;display:block;margin-bottom:3px}
 input,select{width:100%;padding:7px;border:1px solid #bbb;border-radius:6px;box-sizing:border-box}
 button{grid-column:1/-1;padding:12px;background:#1d3557;color:#fff;border:0;border-radius:8px;font-size:1rem;cursor:pointer}
 #out{margin-top:20px;padding:16px;border:1px solid #ddd;border-radius:10px;display:none}
 .bar{height:14px;background:#e5e7eb;border-radius:7px;overflow:hidden}.bar>div{height:100%;background:#2a9d8f}
 li.pos{color:#1b7f6d} li.neg{color:#c2410c}
</style></head><body>
<h2>Colorectal cancer &ndash; 5-year survival prediction (SEER-trained)</h2>
<div class="warn">{{ disclaimer }}</div>
<form id="f">
{% for c, r in numeric.items() %}
 <div><label>{{ c }} (blank = unknown; typical {{ r.min|int }}-{{ r.max|int }})</label><input type="number" step="any" name="{{ c }}" value="{{ r.median|int }}"></div>
{% endfor %}
{% for c, opts in categorical.items() %}
 <div><label>{{ c }}</label><select name="{{ c }}">{% for o in opts %}<option {% if o == 'Unknown' %}selected{% endif %}>{{ o }}</option>{% endfor %}</select></div>
{% endfor %}
 <button type="submit">Predict</button>
</form>
<div id="out"></div>
<script>
document.getElementById('f').addEventListener('submit', async e => {
  e.preventDefault();
  const body = Object.fromEntries(new FormData(e.target).entries());
  const r = await fetch('/predict', {method:'POST', headers:{'Content-Type':'application/json'}, body: JSON.stringify(body)});
  const d = await r.json(), out = document.getElementById('out'); out.style.display = 'block';
  if (!r.ok) { out.innerHTML = '<b>Error:</b> ' + d.error; return; }
  const pct = (d.survival_probability * 100).toFixed(1);
  out.innerHTML = '<h3>' + d.prediction + '</h3><div class="bar"><div style="width:' + pct + '%"></div></div>' +
    '<p>Estimated (calibrated) 5-year overall-survival probability: <b>' + pct + '%</b> (model: ' + d.model + ')</p>' +
    '<b>Top contributing factors</b><ul>' + d.top_factors.map(t => '<li class="' + (t.impact > 0 ? 'pos' : 'neg') + '">' +
    t.feature + ' &mdash; ' + t.direction + ' (' + t.impact + ')</li>').join('') + '</ul><small>' + d.disclaimer + '</small>';
});
</script></body></html>
"""

app = Flask(__name__)


@app.get("/")
def index():
    return render_template_string(PAGE, numeric=NUMERIC, categorical=CATEGORICAL, disclaimer=DISCLAIMER)


@app.post("/predict")
def predict():
    payload = request.get_json(silent=True) or request.form.to_dict()
    try:
        return jsonify(predict_one(payload))
    except (ValueError, KeyError) as e:
        return jsonify({"error": str(e)}), 400


@app.get("/health")
def health():
    return jsonify({"status": "ok", "model": meta["model_name"]})


if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000, debug=False)

In [ ]:
from importlib.metadata import version
req = []
for p in ["flask", "gunicorn", "pandas", "numpy", "scikit-learn", "xgboost", "lightgbm", "shap", "joblib"]:
    try: req.append(f"{p}=={version(p)}")
    except Exception: req.append(p)
Path("requirements.txt").write_text("\n".join(req) + "\n")
Path("Dockerfile").write_text("""FROM python:3.11-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY app.py .
COPY artifacts/ artifacts/
EXPOSE 5000
CMD ["gunicorn", "-b", "0.0.0.0:5000", "-w", "2", "app:app"]
""")
Path(".dockerignore").write_text("Survival_Data.csv\ndata.csv\nmlruns/\nmlflow.db\noutputs/\n*.ipynb\n.ipynb_checkpoints/\n")
print(Path("requirements.txt").read_text())

In [ ]:
# automatic test of the web app (Flask test client - no server needed)
import importlib
sys.path.insert(0, os.getcwd())
import app as crc_app
importlib.reload(crc_app)
client = crc_app.app.test_client()
print("GET /health ->", client.get("/health").get_json()); print("GET /       ->", client.get("/").status_code)

sample_patient = json.loads(X_test[FEATS].iloc[[0]].to_json(orient="records"))[0]
t0 = time.time(); r = client.post("/predict", json=sample_patient)
print("POST /predict ->", r.status_code, f"({time.time()-t0:.1f}s)"); print(json.dumps(r.get_json(), indent=2))
r2 = client.post("/predict", json={"Age_years": 70, "Sex": "Female"})          # incomplete input: missing fields become NaN / Unknown
print("Incomplete input ->", r2.status_code, r2.get_json()["survival_probability"])

### How to run
```bash
python app.py                                   # http://127.0.0.1:5000
docker build -t crc-survival . && docker run -p 5000:5000 crc-survival
mlflow ui --backend-store-uri sqlite:///mlflow.db --port 5001
```

## Step 19 — Auto-filled abstract numbers and the base-paper limitations table

In [ ]:
b = test_table.loc[final_name]
n_all, n_lab = len(C), int(C.y5.notna().sum()); n_surv = int((C.y5 == 1).sum()); n_cens = int(C.y5.isna().sum())
cox_c = surv_table.loc["Cox PH (lifelines)", "C-index"]; xgbc = surv_table.loc["XGBoost-Cox", "C-index"]
top3 = ", ".join(shap_by_feature.head(3).index)
fin_ci = ci_table.loc[final_name]
cal_raw = cal_table.iloc[0] if has_proba else None

print("=" * 90)
print(f"Cohort ................ {n_all:,} patients (first primary colon/rectum cancer, SEER 17 registries, Nov 2025 sub, diagnosed 2010-2018)")
print(f"5-year outcome known .. {n_lab:,}; of whom {n_surv:,} ({100*n_surv/n_lab:.1f}%) survived >= 5 years; {n_cens:,} ({100*n_cens/n_all:.1f}%) censored before 60 months")
print(f"Task .................. primary: five-year overall-survival CLASSIFICATION; secondary: time-to-event survival modelling")
print(f"Prediction time point . Set {FEATURE_SET} = {TIMEPOINT[FEATURE_SET]} ({len(FEATS)} features); Set A (diagnosis) reported as a secondary model")
print(f"Selected model ........ {final_name} (selection rule: max CV ROC-AUC, ties within {TIE_MARGIN} -> min CV {TIE_BREAK}); tied: {', '.join(tied.index)}")
print(f"[ROC-AUC] ............. {fin_ci['ROC-AUC']}   | PR-AUC {fin_ci['PR-AUC']} | Brier {fin_ci['Brier']}")
print(f"[threshold {THRESHOLD}] ....... accuracy {fin_ci['Accuracy']} | sensitivity {fin_ci['Sensitivity']} | specificity {fin_ci['Specificity']} | F1 {fin_ci['F1-score']}")
if has_proba:
    print(f"Calibration ........... intercept {cal_raw['Calib. intercept']:.3f}, slope {cal_raw['Calib. slope']:.3f} (raw); Brier {brier_raw:.3f} -> {brier_iso:.3f} after isotonic calibration")
print(f"[vs traditional] ...... AUC {test_table.loc[TRAD_REF, 'ROC-AUC']:.3f} for logistic regression on stage+age+sex -> gain {np.mean(auc_b - trad_b):+.3f} (95% CI {diff_ci[0]:+.3f} to {diff_ci[1]:+.3f})")
print(f"Strict temporal val. .. {t_name} developed on 2010-{TEMPORAL_CUT} only -> 2016-2018 ROC-AUC {t_ci.iloc[0]['ROC-AUC']}")
print(f"Survival models ....... Cox C-index {cox_c:.3f} | XGBoost-Cox C-index {xgbc:.3f}")
print(f"Feature sets (ROC-AUC). " + " | ".join(f"{k.split(':')[0]} {v:.3f}" for k, v in sens["ROC-AUC"].items()))
print(f"[top three predictors] {top3}   (SHAP; associations used by the model, not causal effects)")
print("=" * 90)

print(f"""
RESULTS PARAGRAPH (draft - check every number against the tables above before using it):
Among {n_all:,} patients, {n_lab:,} had a known five-year outcome, of whom {n_surv:,} ({100*n_surv/n_lab:.1f}%) survived at least five years;
{n_cens:,} ({100*n_cens/n_all:.1f}%) were censored before 60 months. {final_name}, selected by the cross-validation selection rule, achieved a
ROC-AUC of {fin_ci['ROC-AUC']} on the held-out test set (accuracy {fin_ci['Accuracy']}, Brier score {fin_ci['Brier']}); models tied under the rule
({', '.join(tied.index)}) showed near-identical discrimination. The traditional logistic regression on stage, age and sex reached
{test_table.loc[TRAD_REF, 'ROC-AUC']:.3f}. In strict temporal validation (development 2010-{TEMPORAL_CUT}, test 2016-2018, all tuning within the development
period) the ROC-AUC was {t_ci.iloc[0]['ROC-AUC']}. SHAP identified {top3} as the strongest contributors to the model's predictions.""")

### Base-paper limitations → where this notebook addresses them
| Limitation of the base paper | How it is addressed here | Where |
|---|---|---|
| Only binary classification, time-to-event not used | Kaplan-Meier, Cox PH, XGBoost-Cox, optional RSF / GBSA / CoxNet; C-index, time-dependent AUC, IPCW Brier | Step 15 |
| Censored patients dropped (bias) | Censoring quantified in the attrition table; censored patients stay in survival models; IPCW-weighted classifier and IPCW Brier | Steps 4, 15 |
| Feature selection on the whole data, then CV (leakage) | L1 selection is a pipeline step, refitted inside every CV fold | Steps 7, 9 |
| Chi-square is univariate, ignores interactions | Mutual information, L1/Lasso, permutation importance and SHAP compared | Step 8 |
| No explainability beyond a variable ranking | SHAP (global + patient), LIME, partial-dependence plots | Step 16 |
| One dataset, no external validation | Strict temporal validation (selection + tuning only in 2010-15 → test 2016-18); external validation on GENIE BPC CRC in a separate notebook | Step 14 |
| Class imbalance | Class weights, PR-AUC, balanced accuracy, MCC, F1 | Steps 9, 11 |
| No calibration check | Calibration curve, Brier, calibration intercept/slope, out-of-fold isotonic calibrator used by the app; decision-curve analysis | Steps 12, 12b |
| Unclear model choice | Explicit selection rule (post hoc), bootstrap CIs and paired ΔAUC tests | Steps 9, 11b |
| Potentially leaky variable ("probable recurrence") | Death-related `Reason no surgery` excluded; feature sets A/B/C + sensitivity analysis; shuffled-label sanity check | Steps 4, 5, 12, 13 |
| No deployment | Flask app with patient-level SHAP, Docker, MLflow | Steps 17, 18 |
| Tuning not described | RandomizedSearchCV (optional Optuna) | Step 10 |

### Limitations to state in the paper
* SEER provides **all-cause** survival (no cause-specific death in this export) – survival is not cancer-specific.
* SEER **treatment variables** (surgery, radiation, chemotherapy) have documented limitations (agreement you signed) – they are excluded from the main model (feature set B) and only used in the sensitivity analysis.
* Registry data lack comorbidity, performance status, molecular markers (MSI, KRAS/BRAF) and detailed treatment – survival prediction from these variables has an upper limit.
* The 2018+ grade categories are assumed to map to grades I-IV (verify in the SEER dictionary); stage coding changed between 2010 and 2018 and was harmonised.
* Single country (US) population; external validation on a non-US cohort is advisable.
* The web app is a **research prototype**, not a diagnostic device.
* Set B is a **post-surgical** model (node counts encode surgery – see the cross-table in Step 13); Set A is the diagnosis-time model.
* Discrimination is within the range of recent CRC survival studies, but direct comparison is inappropriate because populations, outcome definitions, predictors and validation designs differ.

In [ ]:
print("Reproducibility info")
for k, v in {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__, "scikit-learn": __import__("sklearn").__version__,
             "xgboost": __import__("xgboost").__version__, "lightgbm": __import__("lightgbm").__version__, "shap": shap.__version__,
             "lifelines": __import__("lifelines").__version__, "mlflow": mlflow.__version__, "seed": SEED}.items():
    print(f"  {k:14s} {v}")